# EV-SPARK-Cover v1.3.1 — End-to-End Paper Run with Reviewer Robustness Checks

Phiên bản này kế thừa toàn bộ thuật toán và robustness protocol của v1.3, nhưng dùng **đúng các đường dẫn đã được xác minh trong notebook v1.2**. Chỉ các cache tiền xử lý tương thích được tái sử dụng: extracted data package và ba station–POI feature cache. Mọi benchmark, selected sites, CV, ablation, statistics và paper tables của v1.3.1 vẫn được chạy lại trong namespace riêng.


## Thay đổi riêng của v1.3.1

1. Giữ nguyên MODE, robustness checks, thuật toán, evaluator và paper-ready exports của v1.3.
2. Lấy đúng `DRIVE_ROOT`, `DATA_ZIP` và `OUTPUT_ROOT` từ notebook v1.2 đã chạy.
3. Output mới nằm tại `outputs/EV_SPARK_Cover/v1_3_1/<RUN_MODE>/`.
4. Ưu tiên cache hiện tại của v1.3.1; nếu chưa có thì dùng package đã giải nén và ba station–POI caches của `v1_2/paper/cache`.
5. Mọi cache tái sử dụng đều được kiểm tra manifest hoặc schema/station IDs và ghi vào `audits/cache_reuse_audit.csv`.


In [1]:
# Colab dependency cell — chạy một lần khi runtime mới.
%pip -q install pyarrow scipy scikit-learn matplotlib nbformat

# Early audit before the shared cell_audit helper exists.
import importlib.util as _iu
_required_packages = ['pyarrow', 'scipy', 'sklearn', 'matplotlib', 'nbformat']
_missing_packages = [p for p in _required_packages if _iu.find_spec(p) is None]
assert not _missing_packages, f'Missing dependencies: {_missing_packages}'
print('EARLY CELL AUDIT 01_dependencies: PASS')


EARLY CELL AUDIT 01_dependencies: PASS


In [2]:
from __future__ import annotations

import hashlib
import gc
import heapq
import json
import math
import os
import platform
import random
import shutil
import time
import traceback
import warnings
import zipfile
from dataclasses import dataclass
from importlib.metadata import version as package_version
from pathlib import Path
from typing import Any, Iterable, Sequence

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import Markdown, display
from scipy.optimize import lsq_linear
from scipy.sparse import csr_matrix
from scipy.spatial import cKDTree, distance_matrix
from scipy.stats import spearmanr, wilcoxon
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')

PROJECT_VERSION = '1.3.1'
SEED = 42
RUN_MODE = 'paper'             # 'smoke' | 'balanced' | 'paper'
RESUME = True
FORCE_REBUILD = False
STRICT_CELL_AUDIT = True
STRICT_FINAL_AUDIT = False     # đổi True sau khi đã xem audit_report.csv
CHECKPOINT_EVERY = 10          # ghi CSV sau tối đa 10 run; luôn flush ở cuối mỗi stage

PERIODS = [
    'weekday_morning',
    'weekday_evening',
    'weekend_midday',
    'weekend_evening',
]
PERIOD_COLS = {p: f'period_mean__{p}' for p in PERIODS}
POI_TYPES = [
    'tourism_hotel',
    'market_local',
    'commercial_industrial_area',
    'mall_supermarket',
    'transport_hub',
    'parking',
]
TEST_PROVINCES = ['hanoi', 'hochiminh', 'haiphong', 'danang', 'cantho']

MODE = {
    'smoke': dict(
        demand_radii=[3.0], demand_alphas=[0.1],
        service_radii=[2.5], sigma_ratios=[0.5], taus=[1.0],
        equity_lambdas=[0.0, 1.0], dmins=[0.5], dexistings=[0.5],
        local_passes=[0, 1], budget_fracs=[0.05], tune_budget=0.05,
        random_repeats=2, station_blocks=2, tune_block_ids=[0],
        max_dev_provinces=5, max_test_provinces=1,
        local_candidate_pool=30,
        robustness_block_seeds=[0, 42],
        robustness_partition_modes=['kmeans', 'axis_quantiles'],
        robustness_test_provinces=['hanoi', 'cantho'],
        robustness_lodo_max_days=1,
        robustness_bootstrap_repeats=500,
    ),
    'balanced': dict(
        demand_radii=[1.0, 3.0, 5.0], demand_alphas=[0.0, 0.1, 1.0],
        service_radii=[1.5, 2.5, 4.0], sigma_ratios=[0.35, 0.5, 0.75], taus=[0.5, 1.0, 2.0],
        equity_lambdas=[0.0, 0.5, 1.0, 2.0], dmins=[0.25, 0.5, 1.0], dexistings=[0.25, 0.5, 1.0],
        local_passes=[0, 1, 3], budget_fracs=[0.02, 0.05, 0.10], tune_budget=0.05,
        random_repeats=10, station_blocks=3, tune_block_ids=[0],
        max_dev_provinces=None, max_test_provinces=None,
        local_candidate_pool=60,
        robustness_block_seeds=[0, 1, 2, 42],
        robustness_partition_modes=['kmeans', 'axis_quantiles'],
        robustness_test_provinces=['hanoi', 'hochiminh', 'haiphong', 'danang', 'cantho'],
        robustness_lodo_max_days=3,
        robustness_bootstrap_repeats=2000,
    ),
    'paper': dict(
        demand_radii=[1.0, 3.0, 5.0], demand_alphas=[0.0, 0.01, 0.1, 1.0, 10.0],
        service_radii=[1.5, 2.5, 4.0, 5.0], sigma_ratios=[0.35, 0.5, 0.75], taus=[0.5, 1.0, 2.0],
        equity_lambdas=[0.0, 0.5, 1.0, 2.0], dmins=[0.25, 0.5, 1.0], dexistings=[0.25, 0.5, 1.0],
        local_passes=[0, 1, 3], budget_fracs=[0.02, 0.05, 0.10], tune_budget=0.05,
        random_repeats=30, station_blocks=3, tune_block_ids=[0, 1],
        max_dev_provinces=None, max_test_provinces=None,
        local_candidate_pool=100,
        robustness_block_seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 42],
        robustness_partition_modes=['kmeans', 'axis_quantiles'],
        robustness_test_provinces=['hanoi', 'hochiminh', 'haiphong', 'danang', 'cantho'],
        robustness_lodo_max_days=None,
        robustness_bootstrap_repeats=5000,
    ),
}[RUN_MODE]

rng_global = np.random.default_rng(SEED)
random.seed(SEED)
np.random.seed(SEED)

assert RUN_MODE in {'smoke','balanced','paper'}
assert len(PERIODS) == 4 and len(set(PERIODS)) == 4
assert set(TEST_PROVINCES) == {'hanoi','hochiminh','haiphong','danang','cantho'}
print('PROJECT_VERSION =', PROJECT_VERSION)
print('RUN_MODE =', RUN_MODE)
print(json.dumps(MODE, indent=2))
print('EARLY CELL AUDIT 02_config: PASS')

PROJECT_VERSION = 1.3.1
RUN_MODE = paper
{
  "demand_radii": [
    1.0,
    3.0,
    5.0
  ],
  "demand_alphas": [
    0.0,
    0.01,
    0.1,
    1.0,
    10.0
  ],
  "service_radii": [
    1.5,
    2.5,
    4.0,
    5.0
  ],
  "sigma_ratios": [
    0.35,
    0.5,
    0.75
  ],
  "taus": [
    0.5,
    1.0,
    2.0
  ],
  "equity_lambdas": [
    0.0,
    0.5,
    1.0,
    2.0
  ],
  "dmins": [
    0.25,
    0.5,
    1.0
  ],
  "dexistings": [
    0.25,
    0.5,
    1.0
  ],
  "local_passes": [
    0,
    1,
    3
  ],
  "budget_fracs": [
    0.02,
    0.05,
    0.1
  ],
  "tune_budget": 0.05,
  "random_repeats": 30,
  "station_blocks": 3,
  "tune_block_ids": [
    0,
    1
  ],
  "max_dev_provinces": null,
  "max_test_provinces": null,
  "local_candidate_pool": 100,
  "robustness_block_seeds": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    42
  ],
  "robustness_partition_modes": [
    "kmeans",
    "axis_quantiles"
  ],
  "robustness_test_provinces": [
    "hanoi

## 1. Đường dẫn, namespace v1.3.1 và cache bridge từ v1.2

Các đường dẫn Google Drive dưới đây được lấy trực tiếp từ notebook v1.2 đã chạy. Output v1.3.1 được tách riêng tại `v1_3_1/<RUN_MODE>`. Notebook chỉ tái sử dụng cache đầu vào đã kiểm tra tương thích; không đọc lại kết quả benchmark hoặc hyperparameter outputs của v1.2.


In [3]:
# Exact Google Drive paths copied from the executed v1.2 notebook.
DRIVE_ROOT = Path('/content/drive/MyDrive/DSP391m - AI1905')
DATA_ZIP = DRIVE_ROOT / 'data/EV_SPARK_ICTA2026_Data_20260622.zip'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs/EV_SPARK_Cover'

LOCAL_DATA_ZIP = Path('/mnt/data/EV_SPARK_ICTA2026_Data_20260622.zip')
LOCAL_OUTPUT_ROOT = Path('/mnt/data/EV_SPARK_Cover_outputs')

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
else:
    DATA_ZIP = LOCAL_DATA_ZIP
    OUTPUT_ROOT = LOCAL_OUTPUT_ROOT

# v1.3.1 outputs are isolated from all previous versions.
VERSION_DIRNAME = f'v{PROJECT_VERSION.replace(".", "_")}'
RUN_ROOT = OUTPUT_ROOT / VERSION_DIRNAME / RUN_MODE
CACHE_DIR = RUN_ROOT / 'cache'
RAW_DIR = RUN_ROOT / 'raw_results'
TABLE_DIR = RUN_ROOT / 'tables'
FIGURE_DIR = RUN_ROOT / 'figures'
AUDIT_DIR = RUN_ROOT / 'audits'
CONFIG_DIR = RUN_ROOT / 'config'
for d in [RUN_ROOT, CACHE_DIR, RAW_DIR, TABLE_DIR, FIGURE_DIR, AUDIT_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Safe read-only cache source, verified from the executed v1.2 paper run.
V12_PAPER_ROOT = OUTPUT_ROOT / 'v1_2' / 'paper'
V12_CACHE_DIR = V12_PAPER_ROOT / 'cache'
V12_PACKAGE_DIR = V12_CACHE_DIR / 'data_extract' / 'EV_SPARK_ICTA2026_Data_20260622'
V12_STATION_POI_CACHE = {
    (1.0, 0.5): V12_CACHE_DIR / 'station_poi_features_R1_S0.5.parquet',
    (3.0, 0.5): V12_CACHE_DIR / 'station_poi_features_R3_S0.5.parquet',
    (5.0, 0.5): V12_CACHE_DIR / 'station_poi_features_R5_S0.5.parquet',
}
CACHE_REUSE_ROWS: list[dict[str, Any]] = []

assert all(d.exists() for d in [RUN_ROOT, CACHE_DIR, RAW_DIR, TABLE_DIR, FIGURE_DIR, AUDIT_DIR, CONFIG_DIR])
print('DATA_ZIP          :', DATA_ZIP)
print('RUN_ROOT          :', RUN_ROOT)
print('V12_PAPER_ROOT    :', V12_PAPER_ROOT)
print('V12_PACKAGE_FOUND :', V12_PACKAGE_DIR.exists())
print('V12 feature caches:', {str(k): p.exists() for k, p in V12_STATION_POI_CACHE.items()})
print('EARLY CELL AUDIT 04_paths: PASS')


Mounted at /content/drive
DATA_ZIP          : /content/drive/MyDrive/DSP391m - AI1905/data/EV_SPARK_ICTA2026_Data_20260622.zip
RUN_ROOT          : /content/drive/MyDrive/DSP391m - AI1905/outputs/EV_SPARK_Cover/v1_3_1/paper
V12_PAPER_ROOT    : /content/drive/MyDrive/DSP391m - AI1905/outputs/EV_SPARK_Cover/v1_2/paper
V12_PACKAGE_FOUND : True
V12 feature caches: {'(1.0, 0.5)': True, '(3.0, 0.5)': True, '(5.0, 0.5)': True}
EARLY CELL AUDIT 04_paths: PASS


In [4]:
# ---------- Safe export, checkpoint, cell-audit and reproducibility utilities ----------

CELL_AUDIT_ROWS: list[dict[str, Any]] = []

def _json_default(x: Any):
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return x.tolist()
    raise TypeError(f'Not JSON serializable: {type(x)}')


def atomic_write_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=index, encoding='utf-8-sig')
    os.replace(tmp, path)


def atomic_write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default), encoding='utf-8')
    os.replace(tmp, path)


def atomic_write_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def record_cache_reuse(
    artifact: str,
    source: Path | None,
    destination: Path | None,
    reused: bool,
    compatible: bool,
    detail: str = '',
) -> None:
    CACHE_REUSE_ROWS.append({
        'artifact': artifact,
        'source': str(source) if source is not None else '',
        'destination': str(destination) if destination is not None else '',
        'reused': bool(reused),
        'compatible': bool(compatible),
        'detail': detail,
    })
    atomic_write_csv(pd.DataFrame(CACHE_REUSE_ROWS), AUDIT_DIR / 'cache_reuse_audit.csv')


def checkpoint_rows(rows: list[dict[str, Any]], path: Path, *, force: bool = False,
                    dedup_keys: Sequence[str] | None = None) -> None:
    if not force and len(rows) % CHECKPOINT_EVERY != 0:
        return
    df = pd.DataFrame(rows)
    if dedup_keys and len(df):
        valid = [c for c in dedup_keys if c in df.columns]
        if valid: df = df.drop_duplicates(valid, keep='last')
    atomic_write_csv(df, path)


def cell_audit(cell_id: str, checks: dict[str, bool], details: dict[str, Any] | None = None) -> None:
    details = details or {}
    now = pd.Timestamp.utcnow().isoformat()
    new=[]
    for name, passed in checks.items():
        new.append({'cell_id':cell_id,'check':name,'passed':bool(passed),
                    'detail':str(details.get(name,''))[:2000],'utc':now})
    CELL_AUDIT_ROWS.extend(new)
    atomic_write_csv(pd.DataFrame(CELL_AUDIT_ROWS), AUDIT_DIR/'cell_audit_runtime.csv')
    failed=[r for r in new if not r['passed']]
    if failed and STRICT_CELL_AUDIT:
        raise RuntimeError(f'Cell audit failed [{cell_id}]: {failed}')
    print(f'CELL AUDIT {cell_id}:', 'PASS' if not failed else f'FAIL {failed}')


def sha256_file(path: Path, block: int = 2**20) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(block), b''):
            h.update(chunk)
    return h.hexdigest().upper()


def stable_seed(*parts: Any) -> int:
    s = '|'.join(map(str, parts)).encode('utf-8')
    return int(hashlib.sha256(s).hexdigest()[:8], 16)


def safe_spearman(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, float); p = np.asarray(p, float)
    mask = np.isfinite(y) & np.isfinite(p)
    if mask.sum() < 3 or np.nanstd(y[mask]) == 0 or np.nanstd(p[mask]) == 0:
        return 0.0
    return float(spearmanr(y[mask], p[mask]).statistic)


def normalized_mae(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, float); p = np.asarray(p, float)
    mask = np.isfinite(y) & np.isfinite(p)
    if mask.sum() == 0: return np.nan
    scale = np.mean(np.abs(y[mask])) + 1e-9
    return float(np.mean(np.abs(y[mask] - p[mask])) / scale)

cell_audit('05_utilities', {
    'atomic_csv_callable': callable(atomic_write_csv),
    'atomic_json_callable': callable(atomic_write_json),
    'checkpoint_batch_positive': CHECKPOINT_EVERY >= 1,
    'stable_seed_deterministic': stable_seed('x',1) == stable_seed('x',1),
    'cache_reuse_recorder_callable': callable(record_cache_reuse),
})


def runtime_environment_snapshot() -> dict[str, Any]:
    cpu_model = platform.processor() or 'unknown'
    try:
        for line in Path('/proc/cpuinfo').read_text(errors='ignore').splitlines():
            if line.lower().startswith('model name'):
                cpu_model = line.split(':',1)[1].strip()
                break
    except Exception:
        pass
    memory_gb = None
    try:
        memory_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
    except Exception:
        pass
    packages = {}
    for name in ['numpy','pandas','scipy','scikit-learn','pyarrow','matplotlib','nbformat']:
        try:
            packages[name] = package_version(name)
        except Exception:
            packages[name] = 'unknown'
    return {
        'project_version': PROJECT_VERSION,
        'run_mode': RUN_MODE,
        'python': platform.python_version(),
        'platform': platform.platform(),
        'cpu_model': cpu_model,
        'logical_cpu_count': os.cpu_count(),
        'system_memory_gb': round(memory_gb, 3) if memory_gb is not None else None,
        'packages': packages,
        'runtime_metric_scope': 'selection time only; excludes data loading, feature/kernel construction, evaluation, and export',
    }


RUNTIME_ENVIRONMENT = runtime_environment_snapshot()
atomic_write_json(RUNTIME_ENVIRONMENT, CONFIG_DIR/'runtime_environment.json')


CELL AUDIT 05_utilities: PASS


## 2. Giải nén, xác minh manifest và đọc dữ liệu

`manifest_audit.csv` được export **trước** khi notebook có thể dừng vì checksum/size không khớp.


In [5]:
CURRENT_EXTRACT_ROOT = CACHE_DIR / 'data_extract'
CURRENT_PACKAGE_DIR = CURRENT_EXTRACT_ROOT / 'EV_SPARK_ICTA2026_Data_20260622'

if FORCE_REBUILD and CURRENT_EXTRACT_ROOT.exists():
    shutil.rmtree(CURRENT_EXTRACT_ROOT)

if CURRENT_PACKAGE_DIR.exists() and RESUME and not FORCE_REBUILD:
    PACKAGE_DIR = CURRENT_PACKAGE_DIR
    PACKAGE_SOURCE = 'v1_3_1_cache'
    record_cache_reuse(
        'data_extract_package', CURRENT_PACKAGE_DIR, CURRENT_PACKAGE_DIR,
        reused=True, compatible=True, detail='Current v1.3.1 extracted package reused.'
    )
elif V12_PACKAGE_DIR.exists() and not FORCE_REBUILD:
    # Read the verified v1.2 extracted package directly; do not duplicate the 7-day time-series file.
    PACKAGE_DIR = V12_PACKAGE_DIR
    PACKAGE_SOURCE = 'v1_2_cache'
    record_cache_reuse(
        'data_extract_package', V12_PACKAGE_DIR, None,
        reused=True, compatible=True,
        detail='Read-only reuse of v1.2 extracted package; manifest is re-verified below.'
    )
else:
    if not DATA_ZIP.exists():
        raise FileNotFoundError(
            'Không tìm thấy cả extracted package v1.2/v1.3.1 và ZIP nguồn: '
            f'{DATA_ZIP}'
        )
    CURRENT_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as zf:
        bad = zf.testzip()
        if bad:
            raise RuntimeError(f'ZIP CRC failure: {bad}')
        zf.extractall(CURRENT_EXTRACT_ROOT)
    PACKAGE_DIR = CURRENT_PACKAGE_DIR
    PACKAGE_SOURCE = 'fresh_zip_extract'
    record_cache_reuse(
        'data_extract_package', DATA_ZIP, CURRENT_PACKAGE_DIR,
        reused=False, compatible=True, detail='Fresh extraction from source ZIP.'
    )

print('PACKAGE_DIR   :', PACKAGE_DIR)
print('PACKAGE_SOURCE:', PACKAGE_SOURCE)

manifest = json.loads((PACKAGE_DIR / 'manifest.json').read_text(encoding='utf-8-sig'))
manifest_rows = []
for item in manifest.get('files', []):
    p = PACKAGE_DIR / item['file']
    manifest_rows.append({
        'file': item['file'],
        'exists': p.exists(),
        'expected_bytes': item['bytes'],
        'actual_bytes': p.stat().st_size if p.exists() else -1,
        'size_ok': p.exists() and p.stat().st_size == item['bytes'],
        'sha256_ok': p.exists() and sha256_file(p) == item['sha256'],
    })
manifest_audit = pd.DataFrame(manifest_rows)
atomic_write_csv(manifest_audit, AUDIT_DIR / 'manifest_audit.csv')
if len(manifest_audit) and not (manifest_audit['size_ok'] & manifest_audit['sha256_ok']).all():
    raise RuntimeError('Data manifest verification failed. Xem audits/manifest_audit.csv')

candidates_all = pq.read_table(PACKAGE_DIR / 'candidate_features.parquet').to_pandas(ignore_metadata=True)
pois_all = pq.read_table(PACKAGE_DIR / 'poi_nodes.parquet').to_pandas(ignore_metadata=True)
stations_all = pq.read_table(PACKAGE_DIR / 'station_model_temporal.parquet').to_pandas(ignore_metadata=True)
station_catalog = pq.read_table(PACKAGE_DIR / 'station_catalog_from_kml.parquet').to_pandas(ignore_metadata=True)
protocol_old = json.loads((PACKAGE_DIR / 'benchmark_protocol.json').read_text(encoding='utf-8-sig'))

candidates = candidates_all.loc[candidates_all['optimization_eligible'].eq(True)].copy()
pois = pois_all.loc[pois_all['poi_type'].isin(POI_TYPES)].copy()
stations_strict = stations_all.loc[stations_all['strict_7d_eligible'].eq(True)].copy()

inventory = pd.DataFrame([
    {'table':'package_source', 'rows':0, 'columns':0, 'source':PACKAGE_SOURCE},
    {'table':'candidate_features_all', 'rows':len(candidates_all), 'columns':candidates_all.shape[1]},
    {'table':'candidate_eligible', 'rows':len(candidates), 'columns':candidates.shape[1]},
    {'table':'poi_nodes_supported_types', 'rows':len(pois), 'columns':pois.shape[1]},
    {'table':'stations_all', 'rows':len(stations_all), 'columns':stations_all.shape[1]},
    {'table':'stations_strict_temporal', 'rows':len(stations_strict), 'columns':stations_strict.shape[1]},
])
province_inventory = (
    pd.concat([
        candidates.groupby('province_key').size().rename('eligible_candidates'),
        pois.groupby('province_key').size().rename('pois'),
        stations_all.groupby('province_key').size().rename('stations_all'),
        stations_strict.groupby('province_key').size().rename('stations_strict'),
    ], axis=1).fillna(0).reset_index()
)
atomic_write_csv(inventory, TABLE_DIR / 'data_inventory.csv')
atomic_write_csv(province_inventory, TABLE_DIR / 'province_inventory.csv')
display(inventory)
display(province_inventory.head())


required_candidate_cols={'candidate_id','province_key','latitude','longitude','optimization_eligible'}
required_poi_cols={'poi_id','province_key','poi_type','latitude','longitude'}
required_station_cols={'station_id','province_key','latitude','longitude','total_ports','strict_7d_eligible',*PERIOD_COLS.values()}
cell_audit('07_data_load', {
    'manifest_verified': bool((manifest_audit['size_ok'] & manifest_audit['sha256_ok']).all()),
    'candidate_schema': required_candidate_cols.issubset(candidates_all.columns),
    'poi_schema': required_poi_cols.issubset(pois_all.columns),
    'station_schema': required_station_cols.issubset(stations_all.columns),
    'eligible_candidate_nonempty': len(candidates) > 0,
    'strict_station_nonempty': len(stations_strict) > 0,
    'supported_poi_types_only': set(pois.poi_type).issubset(set(POI_TYPES)),
}, {
    'eligible_candidate_nonempty': len(candidates),
    'strict_station_nonempty': len(stations_strict),
})

PACKAGE_DIR   : /content/drive/MyDrive/DSP391m - AI1905/outputs/EV_SPARK_Cover/v1_2/paper/cache/data_extract/EV_SPARK_ICTA2026_Data_20260622
PACKAGE_SOURCE: v1_2_cache


,table,rows,columns,source
0,package_source,0,0,v1_2_cache
1,candidate_features_all,19125,55,NaN
2,candidate_eligible,5509,55,NaN
3,poi_nodes_supported_types,36081,19,NaN
4,stations_all,3813,77,NaN
5,stations_strict_temporal,3811,77,NaN


,province_key,eligible_candidates,pois,stations_all,stations_strict
0,angiang,237,1092,91,91
1,bacninh,265,942,109,109
2,camau,22,121,43,42
3,cantho,171,913,72,72
4,caobang,49,306,21,21


CELL AUDIT 07_data_load: PASS


## 2A. Cache ngày địa phương cho kiểm tra độ bền theo thời gian

Tổng hợp 8.97 triệu quan sát theo từng trạm, ngày địa phương và khung thời gian ngay sau khi nạp dữ liệu. Việc này giữ mức dùng bộ nhớ thấp trước khi các kernel không gian được tạo.


In [6]:
# Build the daily cache early, before spatial kernels and province contexts consume memory.
EARLY_DAILY_CACHE = CACHE_DIR/'usage_daily_period_aggregates.parquet'
EARLY_RECON_AUDIT = AUDIT_DIR/'temporal_reconstruction_audit.csv'


def _period_table_from_daily_early(daily: pd.DataFrame) -> pd.DataFrame:
    sums = daily.groupby(['location_id','period'], as_index=False)[['weighted_sum','weight_s']].sum()
    sums['period_mean'] = sums['weighted_sum']/sums['weight_s'].replace(0,np.nan)
    wide = sums.pivot(index='location_id', columns='period', values='period_mean').reset_index()
    return wide.rename(columns={
        'location_id':'evcs_location_id',
        **{p:PERIOD_COLS[p] for p in PERIODS},
    })[['evcs_location_id', *PERIOD_COLS.values()]]


def precompute_usage_daily_cache() -> tuple[pd.DataFrame, pd.DataFrame]:
    if EARLY_DAILY_CACHE.exists() and EARLY_RECON_AUDIT.exists() and RESUME and not FORCE_REBUILD:
        daily = pd.read_parquet(EARLY_DAILY_CACHE)
        audit = pd.read_csv(EARLY_RECON_AUDIT)
        print(f'Early temporal cache reused: {len(daily):,} rows')
        return daily, audit

    raw_path = PACKAGE_DIR/'evcs_usage_timeseries_7d.parquet'
    pf = pq.ParquetFile(raw_path)
    chunks = []
    print(f'Early temporal aggregation: {pf.num_row_groups} station row groups', flush=True)
    for rg in range(pf.num_row_groups):
        ts = pf.read_row_group(rg, columns=['location_id','timestamp_utc','charging_count']).to_pandas(ignore_metadata=True)
        ts['timestamp_utc'] = pd.to_datetime(ts['timestamp_utc'], utc=True)
        ts = ts.sort_values('timestamp_utc', kind='mergesort').reset_index(drop=True)
        next_ts = ts['timestamp_utc'].shift(-1)
        weight_s = (next_ts-ts['timestamp_utc']).dt.total_seconds().clip(lower=0, upper=1800).fillna(0).to_numpy(float)
        local = ts['timestamp_utc'].dt.tz_convert('Asia/Ho_Chi_Minh')
        hour = local.dt.hour.to_numpy(np.int16)
        weekend = local.dt.dayofweek.to_numpy(np.int8) >= 5
        slot = np.full(len(ts), 3, dtype=np.int8)
        slot[(hour>=6)&(hour<10)] = 0
        slot[(hour>=10)&(hour<16)] = 1
        slot[(hour>=16)&(hour<22)] = 2
        tmp = pd.DataFrame({
            'location_id': ts['location_id'].iloc[0],
            'local_date': local.dt.floor('D').dt.tz_localize(None).to_numpy(dtype='datetime64[D]'),
            'period_code': (weekend.astype(np.int8)*4 + slot).astype(np.int8),
            'weighted_sum': pd.to_numeric(ts.charging_count, errors='coerce').fillna(0).to_numpy(float)*weight_s,
            'weight_s': weight_s,
            'n_points': np.ones(len(ts), dtype=np.int32),
        })
        chunks.append(tmp.groupby(['location_id','local_date','period_code'], as_index=False, sort=False, observed=True)
                      .agg(weighted_sum=('weighted_sum','sum'), weight_s=('weight_s','sum'), n_points=('n_points','sum')))
        if (rg+1) % 1000 == 0:
            print(f'  aggregated {rg+1}/{pf.num_row_groups}', flush=True)

    daily = pd.concat(chunks, ignore_index=True)
    code_to_period = {
        0:'weekday_morning', 1:'weekday_midday', 2:'weekday_evening', 3:'weekday_night',
        4:'weekend_morning', 5:'weekend_midday', 6:'weekend_evening', 7:'weekend_night',
    }
    daily['period'] = daily['period_code'].map(code_to_period)
    daily['local_date'] = pd.to_datetime(daily['local_date']).dt.strftime('%Y-%m-%d')
    daily = daily.drop(columns='period_code')
    atomic_write_parquet(daily, EARLY_DAILY_CACHE)

    reconstructed = _period_table_from_daily_early(daily).set_index('evcs_location_id')
    source = stations_all.set_index('evcs_location_id')
    checks = []
    for p in PERIODS:
        col = PERIOD_COLS[p]
        err = (source[col]-reconstructed[col].reindex(source.index)).abs()
        checks.append({
            'period':p,
            'rows_compared':int(err.notna().sum()),
            'max_abs_error':float(err.max()),
            'mean_abs_error':float(err.mean()),
        })
    audit = pd.DataFrame(checks)
    atomic_write_csv(audit, EARLY_RECON_AUDIT)
    del chunks, ts, local, next_ts, tmp
    gc.collect()
    try:
        import ctypes
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass
    print(f'Early temporal cache written: {len(daily):,} rows')
    return daily, audit


_usage_daily_early, _temporal_reconstruction_early = precompute_usage_daily_cache()
cell_audit('07b_temporal_daily_cache', {
    'daily_cache_exists': EARLY_DAILY_CACHE.exists(),
    'reconstruction_audit_exists': EARLY_RECON_AUDIT.exists(),
    'reconstruction_exact': bool((_temporal_reconstruction_early.max_abs_error < 1e-9).all()),
    'all_complete_periods_present': set(PERIODS).issubset(set(_usage_daily_early.period)),
})
del _usage_daily_early, _temporal_reconstruction_early
gc.collect()
try:
    import ctypes
    ctypes.CDLL('libc.so.6').malloc_trim(0)
except Exception:
    pass


Early temporal aggregation: 3813 station row groups
  aggregated 1000/3813
  aggregated 2000/3813
  aggregated 3000/3813
Early temporal cache written: 114,168 rows
CELL AUDIT 07b_temporal_daily_cache: PASS


## 3. Protocol CV đã khóa

- Năm tỉnh test không được dùng để tuning.
- 29 tỉnh development được chia thành 5 nhóm theo tỉnh (*grouped spatial cross-validation*).
- Trong mỗi tỉnh validation/test, trạm hiện hữu còn được chia thành các khối không gian để tạo benchmark ẩn-trạm (*retrospective station recovery*).


In [7]:
DEV_FOLDS = {
    1: ['hue', 'hungyen', 'lamdong', 'sonla', 'tayninh'],
    2: ['caobang', 'daklak', 'dienbien', 'dongnai', 'nghean', 'ninhbinh'],
    3: ['bacninh', 'dongthap', 'langson', 'laocai', 'quangngai', 'thanhhoa'],
    4: ['angiang', 'camau', 'hatinh', 'quangninh', 'quangtri', 'tuyenquang'],
    5: ['gialai', 'khanhhoa', 'laichau', 'phutho', 'thainguyen', 'vinhlong'],
}
ALL_DEV_PROVINCES = [p for xs in DEV_FOLDS.values() for p in xs]

available = set(stations_all['province_key']) & set(candidates['province_key']) & set(pois['province_key'])
missing_dev = sorted(set(ALL_DEV_PROVINCES) - available)
missing_test = sorted(set(TEST_PROVINCES) - available)
if missing_dev or missing_test:
    raise RuntimeError(f'Missing provinces. dev={missing_dev}, test={missing_test}')

if MODE['max_dev_provinces']:
    # Round-robin sampling preserves all outer folds in smoke mode.
    chosen=[]
    depth=0
    while len(chosen)<MODE['max_dev_provinces']:
        added=False
        for k in sorted(DEV_FOLDS):
            if depth < len(DEV_FOLDS[k]) and len(chosen)<MODE['max_dev_provinces']:
                chosen.append(DEV_FOLDS[k][depth]); added=True
        if not added: break
        depth += 1
    keep=set(chosen)
    DEV_FOLDS_RUN={k:[p for p in v if p in keep] for k,v in DEV_FOLDS.items()}
    DEV_FOLDS_RUN={k:v for k,v in DEV_FOLDS_RUN.items() if v}
else:
    DEV_FOLDS_RUN=DEV_FOLDS

TEST_PROVINCES_RUN = TEST_PROVINCES[:MODE['max_test_provinces']] if MODE['max_test_provinces'] else TEST_PROVINCES

LOCKED_PROTOCOL = {
    'project':'EV-SPARK-Cover', 'version':PROJECT_VERSION, 'seed':SEED, 'run_mode':RUN_MODE,
    'data_zip':str(DATA_ZIP), 'package_dir':str(PACKAGE_DIR), 'package_source':PACKAGE_SOURCE,
    'v1_2_cache_root':str(V12_CACHE_DIR), 'periods':PERIODS, 'poi_types':POI_TYPES,
    'development_folds':DEV_FOLDS_RUN, 'test_provinces':TEST_PROVINCES_RUN,
    'candidate_filter':'optimization_eligible == True',
    'strict_temporal_filter':'strict_7d_eligible == True',
    'mode_config':MODE,
    'rules':[
        'test provinces never used for hyperparameter tuning',
        'all baselines use the same visible network, candidate eligibility, budget, and evaluator',
        'results are exported before final audit and plots',
        'no CatBoost, GNN, MILP, or old teacher labels are used',
        'v1.3.1 result namespace is isolated; only manifest-verified input caches from v1.2 may be reused',
        'true no-temporal and true no-existing-network ablations are required',
        'robustness seeds and alternative partitions are descriptive checks and are never used for tuning',
        'leave-one-local-day-out analyses keep hyperparameters frozen and do not create pseudo-replicated p-values',
        'candidate-availability metrics are geometric diagnostics, not feasible budgeted optima',
        'reported runtime covers site selection only',
    ],
}
atomic_write_json(LOCKED_PROTOCOL, CONFIG_DIR / 'locked_protocol.json')
print(json.dumps(LOCKED_PROTOCOL, ensure_ascii=False, indent=2)[:6000])

cell_audit('09_protocol', {
    'all_dev_unique': len(ALL_DEV_PROVINCES)==len(set(ALL_DEV_PROVINCES)),
    'test_disjoint_dev': set(TEST_PROVINCES_RUN).isdisjoint(set(ALL_DEV_PROVINCES)),
    'all_requested_provinces_available': not missing_dev and not missing_test,
    'paper_has_two_tuning_blocks': RUN_MODE!='paper' or MODE['tune_block_ids']==[0,1],
})

{
  "project": "EV-SPARK-Cover",
  "version": "1.3.1",
  "seed": 42,
  "run_mode": "paper",
  "data_zip": "/content/drive/MyDrive/DSP391m - AI1905/data/EV_SPARK_ICTA2026_Data_20260622.zip",
  "package_dir": "/content/drive/MyDrive/DSP391m - AI1905/outputs/EV_SPARK_Cover/v1_2/paper/cache/data_extract/EV_SPARK_ICTA2026_Data_20260622",
  "package_source": "v1_2_cache",
  "v1_2_cache_root": "/content/drive/MyDrive/DSP391m - AI1905/outputs/EV_SPARK_Cover/v1_2/paper/cache",
  "periods": [
    "weekday_morning",
    "weekday_evening",
    "weekend_midday",
    "weekend_evening"
  ],
  "poi_types": [
    "tourism_hotel",
    "market_local",
    "commercial_industrial_area",
    "mall_supermarket",
    "transport_hub",
    "parking"
  ],
  "development_folds": {
    "1": [
      "hue",
      "hungyen",
      "lamdong",
      "sonla",
      "tayninh"
    ],
    "2": [
      "caobang",
      "daklak",
      "dienbien",
      "dongnai",
      "nghean",
      "ninhbinh"
    ],
    "3": [
      "bac

## 4. Hình học không gian và ma trận soft-count

Khoảng cách được tính trong hệ tọa độ phẳng cục bộ theo từng tỉnh. Với phạm vi một tỉnh/thành, xấp xỉ này đủ chính xác cho bán kính vài kilomet và nhanh hơn tính Haversine cho toàn bộ cặp điểm.


In [8]:
EARTH_RADIUS_KM = 6371.0088


def project_latlon(lat: np.ndarray, lon: np.ndarray, lat0: float | None = None) -> tuple[np.ndarray, float]:
    lat = np.asarray(lat, float); lon = np.asarray(lon, float)
    if lat0 is None: lat0 = float(np.nanmedian(lat))
    x = EARTH_RADIUS_KM * np.deg2rad(lon) * math.cos(math.radians(lat0))
    y = EARTH_RADIUS_KM * np.deg2rad(lat)
    return np.column_stack([x, y]), lat0


def project_with_lat0(lat: np.ndarray, lon: np.ndarray, lat0: float) -> np.ndarray:
    lat = np.asarray(lat, float); lon = np.asarray(lon, float)
    return np.column_stack([
        EARTH_RADIUS_KM * np.deg2rad(lon) * math.cos(math.radians(lat0)),
        EARTH_RADIUS_KM * np.deg2rad(lat),
    ])


def sparse_radial_kernel(src_xy: np.ndarray, dst_xy: np.ndarray, radius_km: float, sigma_km: float) -> csr_matrix:
    if len(src_xy) == 0 or len(dst_xy) == 0:
        return csr_matrix((len(src_xy), len(dst_xy)), dtype=np.float32)
    tree = cKDTree(dst_xy)
    rows, cols, vals = [], [], []
    for i, pt in enumerate(src_xy):
        js = tree.query_ball_point(pt, r=radius_km)
        if not js: continue
        arr = np.asarray(js, dtype=int)
        d = np.linalg.norm(dst_xy[arr] - pt, axis=1)
        w = np.exp(-np.square(d / max(sigma_km, 1e-9)))
        rows.extend([i] * len(arr)); cols.extend(arr.tolist()); vals.extend(w.tolist())
    return csr_matrix((np.asarray(vals, np.float32), (rows, cols)), shape=(len(src_xy), len(dst_xy)))


def poi_type_onehot(poi_df: pd.DataFrame) -> csr_matrix:
    type_to_idx = {t:i for i,t in enumerate(POI_TYPES)}
    cols = poi_df['poi_type'].map(type_to_idx).to_numpy()
    rows = np.arange(len(poi_df))
    vals = np.ones(len(poi_df), dtype=np.float32)
    return csr_matrix((vals, (rows, cols)), shape=(len(poi_df), len(POI_TYPES)))


def build_station_poi_features(radius_km: float, sigma_ratio: float = 0.5) -> pd.DataFrame:
    filename = f'station_poi_features_R{radius_km:g}_S{sigma_ratio:g}.parquet'
    cache = CACHE_DIR / filename
    expected_feature_cols = {f'poi_soft__{t}' for t in POI_TYPES}
    expected_ids = set(stations_strict['station_id'].astype(str))

    if cache.exists() and RESUME and not FORCE_REBUILD:
        result = pd.read_parquet(cache)
        compatible = (
            {'station_id', 'province_key', *expected_feature_cols}.issubset(result.columns)
            and set(result['station_id'].astype(str)) == expected_ids
        )
        if compatible:
            record_cache_reuse(
                filename, cache, cache, reused=True, compatible=True,
                detail='Current v1.3.1 station–POI cache reused.'
            )
            return result
        record_cache_reuse(
            filename, cache, cache, reused=False, compatible=False,
            detail='Current cache failed schema/station-ID compatibility and will be rebuilt.'
        )

    legacy = V12_STATION_POI_CACHE.get((float(radius_km), float(sigma_ratio)))
    if legacy is not None and legacy.exists() and not FORCE_REBUILD:
        legacy_df = pd.read_parquet(legacy)
        compatible = (
            {'station_id', 'province_key', *expected_feature_cols}.issubset(legacy_df.columns)
            and set(legacy_df['station_id'].astype(str)) == expected_ids
        )
        if compatible:
            atomic_write_parquet(legacy_df, cache)
            record_cache_reuse(
                filename, legacy, cache, reused=True, compatible=True,
                detail='Compatible v1.2 station–POI cache copied into the v1.3.1 namespace.'
            )
            return legacy_df
        record_cache_reuse(
            filename, legacy, cache, reused=False, compatible=False,
            detail='v1.2 cache exists but failed schema/station-ID compatibility; rebuilding.'
        )

    out=[]
    for prov in sorted(set(ALL_DEV_PROVINCES + TEST_PROVINCES)):
        sdf = stations_strict.loc[stations_strict.province_key.eq(prov)].reset_index(drop=True)
        pdf = pois.loc[pois.province_key.eq(prov)].reset_index(drop=True)
        if len(sdf)==0 or len(pdf)==0:
            continue
        all_lat = np.r_[sdf.latitude.to_numpy(float), pdf.latitude.to_numpy(float)]
        lat0=float(np.nanmedian(all_lat))
        sxy=project_with_lat0(sdf.latitude, sdf.longitude, lat0)
        pxy=project_with_lat0(pdf.latitude, pdf.longitude, lat0)
        K=sparse_radial_kernel(sxy, pxy, radius_km, radius_km*sigma_ratio)
        X=(K @ poi_type_onehot(pdf)).toarray()
        part=pd.DataFrame(X, columns=[f'poi_soft__{t}' for t in POI_TYPES])
        part.insert(0,'station_id',sdf.station_id.to_numpy())
        part.insert(1,'province_key',prov)
        out.append(part)

    result=pd.concat(out, ignore_index=True)
    atomic_write_parquet(result, cache)
    record_cache_reuse(
        filename, None, cache, reused=False, compatible=True,
        detail='Station–POI features rebuilt under v1.3.1.'
    )
    return result


# Geometry sanity: zero distance -> weight 1; outside radius -> no edge.
_geom_src=np.array([[0.0,0.0],[10.0,0.0]])
_geom_dst=np.array([[0.0,0.0]])
_geom_k=sparse_radial_kernel(_geom_src,_geom_dst,1.0,0.5).toarray()
cell_audit('11_geometry', {
    'kernel_shape': _geom_k.shape==(2,1),
    'zero_distance_weight_one': abs(_geom_k[0,0]-1.0)<1e-6,
    'outside_radius_zero': _geom_k[1,0]==0.0,
    'projection_finite': np.isfinite(project_latlon(np.array([21.0]),np.array([105.8]))[0]).all(),
})

CELL AUDIT 11_geometry: PASS


## 5. Demand model: non-negative ridge regression

Mỗi period có một mô hình riêng. Target là `log1p(period_mean / total_ports)`. Các hệ số POI bị chặn không âm để giữ diễn giải hợp lý.


In [9]:
@dataclass
class DemandModel:
    period: str
    intercept: float
    coef: np.ndarray
    alpha: float
    radius_km: float
    winsor_p99: float

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.clip(self.intercept + X @ self.coef, 0.0, None)


def fit_nonnegative_ridge(X: np.ndarray, y: np.ndarray, alpha: float) -> tuple[float, np.ndarray]:
    X=np.asarray(X,float); y=np.asarray(y,float)
    A=np.column_stack([np.ones(len(X)), X])
    b=y.copy()
    if alpha>0:
        penalty=np.zeros((X.shape[1], X.shape[1]+1), dtype=float)
        penalty[:,1:]=np.eye(X.shape[1]) * math.sqrt(alpha)
        A=np.vstack([A, penalty]); b=np.r_[b, np.zeros(X.shape[1])]
    lower=np.r_[-np.inf, np.zeros(X.shape[1])]
    upper=np.full(X.shape[1]+1, np.inf)
    res=lsq_linear(A,b,bounds=(lower,upper),lsmr_tol='auto',max_iter=300)
    if not res.success:
        raise RuntimeError(f'Non-negative ridge failed: {res.message}')
    return float(res.x[0]), np.asarray(res.x[1:],float)


def make_target(df: pd.DataFrame, period: str, p99: float | None = None) -> tuple[np.ndarray,float]:
    activity=pd.to_numeric(df[PERIOD_COLS[period]],errors='coerce').fillna(0).to_numpy(float)
    ports=np.maximum(pd.to_numeric(df['total_ports'],errors='coerce').fillna(1).to_numpy(float),1.0)
    raw=np.log1p(np.maximum(activity,0)/ports)
    if p99 is None: p99=float(np.nanquantile(raw,0.99))
    return np.minimum(raw,p99),p99


def demand_cv() -> pd.DataFrame:
    path=RAW_DIR/'demand_cv_raw.csv'
    rows=pd.read_csv(path).to_dict('records') if path.exists() and RESUME else []
    done={(float(r['radius_km']),float(r['alpha']),int(r['fold']),str(r['period'])) for r in rows if r.get('status')=='ok'}
    for radius in MODE['demand_radii']:
        feat=build_station_poi_features(radius)
        merged=stations_strict.merge(feat,on=['station_id','province_key'],how='inner')
        fcols=[f'poi_soft__{t}' for t in POI_TYPES]
        for alpha in MODE['demand_alphas']:
            for fold,val_provs in DEV_FOLDS_RUN.items():
                train_provs=[p for k,xs in DEV_FOLDS_RUN.items() if k!=fold for p in xs]
                tr=merged.loc[merged.province_key.isin(train_provs)]
                va=merged.loc[merged.province_key.isin(val_provs)]
                if len(tr)==0 or len(va)==0: continue
                for period in PERIODS:
                    key=(float(radius),float(alpha),int(fold),period)
                    if key in done: continue
                    try:
                        ytr,p99=make_target(tr,period)
                        yva,_=make_target(va,period,p99)
                        intercept,coef=fit_nonnegative_ridge(tr[fcols].to_numpy(float),ytr,alpha)
                        pred=np.clip(intercept+va[fcols].to_numpy(float)@coef,0,None)
                        province_scores=[]; province_nmae=[]
                        for prov,g in va.assign(_y=yva,_p=pred).groupby('province_key'):
                            province_scores.append(safe_spearman(g._y,g._p))
                            province_nmae.append(normalized_mae(g._y,g._p))
                        rows.append({
                            'radius_km':radius,'alpha':alpha,'fold':fold,'period':period,'status':'ok',
                            'n_train':len(tr),'n_val':len(va),'macro_spearman':np.mean(province_scores),
                            'macro_nmae':np.mean(province_nmae),'intercept':intercept,
                            **{f'beta__{t}':coef[i] for i,t in enumerate(POI_TYPES)},
                        })
                    except Exception as exc:
                        rows.append({'radius_km':radius,'alpha':alpha,'fold':fold,'period':period,'status':'error','error':repr(exc)})
                    checkpoint_rows(rows,path)
    checkpoint_rows(rows,path,force=True)
    return pd.DataFrame(rows)


demand_cv_raw=demand_cv()
# RAW EXPORT ALREADY EXISTS before any selection/audit below.
display(demand_cv_raw.head())

cell_audit('13_demand_cv', {
    'raw_export_exists': (RAW_DIR/'demand_cv_raw.csv').exists(),
    'all_expected_periods_present': set(demand_cv_raw.loc[demand_cv_raw.status.eq('ok'),'period'])==set(PERIODS),
    'no_test_province_columns_or_leakage': set(TEST_PROVINCES_RUN).isdisjoint(set(ALL_DEV_PROVINCES)),
    'ridge_coefficients_nonnegative': bool((demand_cv_raw.filter(like='beta__').fillna(0)>=-1e-9).all().all()),
    'no_demand_cv_errors': not demand_cv_raw.status.eq('error').any(),
}, {'no_demand_cv_errors': int(demand_cv_raw.status.eq('error').sum())})

,radius_km,alpha,fold,period,status,n_train,n_val,macro_spearman,macro_nmae,intercept,beta__tourism_hotel,beta__market_local,beta__commercial_industrial_area,beta__mall_supermarket,beta__transport_hub,beta__parking
0,1.0,0.0,1,weekday_morning,ok,2033,496,0.130742,0.736101,0.178056,4.656891e-11,3.922608e-14,0.008048,0.008027,0.015955,1.854999e-21
1,1.0,0.0,1,weekday_evening,ok,2033,496,0.137679,0.736231,0.230594,4.818000e-04,1.861443e-15,0.007254,0.011676,0.012855,7.022354e-29
2,1.0,0.0,1,weekend_midday,ok,2033,496,0.104495,0.691485,0.221385,4.889378e-16,1.448570e-03,0.006131,0.017755,0.027163,7.680518e-24
3,1.0,0.0,1,weekend_evening,ok,2033,496,0.111393,0.719936,0.253653,6.401744e-04,5.607583e-04,0.008660,0.012383,0.011396,1.072957e-21
4,1.0,0.0,2,weekday_morning,ok,2007,522,0.071428,0.714159,0.172523,1.406558e-04,1.840900e-14,0.001829,0.003280,0.017919,2.788719e-03


CELL AUDIT 13_demand_cv: PASS


In [10]:
def select_demand_config(cv: pd.DataFrame) -> dict[str,float]:
    ok=cv.loc[cv.status.eq('ok')].copy()
    agg=(ok.groupby(['radius_km','alpha'],as_index=False)
         .agg(mean_spearman=('macro_spearman','mean'),mean_nmae=('macro_nmae','mean'),rows=('period','size'))
         .sort_values(['mean_spearman','mean_nmae'],ascending=[False,True]))
    if len(agg)==0: raise RuntimeError('No valid demand CV configuration')
    atomic_write_csv(agg,TABLE_DIR/'demand_cv_summary.csv')
    best=agg.iloc[0]
    return {'radius_km':float(best.radius_km),'alpha':float(best.alpha),
            'mean_spearman':float(best.mean_spearman),'mean_nmae':float(best.mean_nmae)}

BEST_DEMAND=select_demand_config(demand_cv_raw)
atomic_write_json(BEST_DEMAND,CONFIG_DIR/'best_demand_config.json')
print('BEST_DEMAND',BEST_DEMAND)

cell_audit('14_select_demand', {
    'best_radius_in_grid': BEST_DEMAND['radius_km'] in MODE['demand_radii'],
    'best_alpha_in_grid': BEST_DEMAND['alpha'] in MODE['demand_alphas'],
    'summary_exported': (TABLE_DIR/'demand_cv_summary.csv').exists(),
    'config_exported': (CONFIG_DIR/'best_demand_config.json').exists(),
    'finite_selection_metrics': np.isfinite([BEST_DEMAND['mean_spearman'],BEST_DEMAND['mean_nmae']]).all(),
})

BEST_DEMAND {'radius_km': 1.0, 'alpha': 0.1, 'mean_spearman': 0.027344646006442534, 'mean_nmae': 0.7445186828337895}
CELL AUDIT 14_select_demand: PASS


In [11]:
def fit_demand_bundle(train_provinces: Sequence[str], radius_km: float, alpha: float) -> dict[str,DemandModel]:
    feat=build_station_poi_features(radius_km)
    merged=stations_strict.merge(feat,on=['station_id','province_key'],how='inner')
    tr=merged.loc[merged.province_key.isin(train_provinces)]
    fcols=[f'poi_soft__{t}' for t in POI_TYPES]
    bundle={}
    for period in PERIODS:
        y,p99=make_target(tr,period)
        intercept,coef=fit_nonnegative_ridge(tr[fcols].to_numpy(float),y,alpha)
        bundle[period]=DemandModel(period,intercept,coef,alpha,radius_km,p99)
    return bundle


def demand_bundle_to_frame(bundle: dict[str,DemandModel], tag: str) -> pd.DataFrame:
    rows=[]
    for p,m in bundle.items():
        rows.append({'tag':tag,'period':p,'intercept':m.intercept,'alpha':m.alpha,'radius_km':m.radius_km,
                     'winsor_p99':m.winsor_p99,**{f'beta__{t}':m.coef[i] for i,t in enumerate(POI_TYPES)}})
    return pd.DataFrame(rows)

FOLD_DEMAND_BUNDLES={}
coef_rows=[]
for fold,val in DEV_FOLDS_RUN.items():
    train=[p for k,xs in DEV_FOLDS_RUN.items() if k!=fold for p in xs]
    b=fit_demand_bundle(train,BEST_DEMAND['radius_km'],BEST_DEMAND['alpha'])
    FOLD_DEMAND_BUNDLES[fold]=b
    coef_rows.append(demand_bundle_to_frame(b,f'fold_{fold}'))
FINAL_DEMAND_BUNDLE=fit_demand_bundle(ALL_DEV_PROVINCES,BEST_DEMAND['radius_km'],BEST_DEMAND['alpha'])
coef_rows.append(demand_bundle_to_frame(FINAL_DEMAND_BUNDLE,'all_development'))
demand_coefficients=pd.concat(coef_rows,ignore_index=True)
atomic_write_csv(demand_coefficients,TABLE_DIR/'demand_coefficients.csv')
display(demand_coefficients)

cell_audit('15_demand_bundles', {
    'five_fold_bundles': len(FOLD_DEMAND_BUNDLES)==len(DEV_FOLDS_RUN),
    'all_periods_each_bundle': all(set(b)==set(PERIODS) for b in FOLD_DEMAND_BUNDLES.values()),
    'final_bundle_all_periods': set(FINAL_DEMAND_BUNDLE)==set(PERIODS),
    'all_coefficients_nonnegative': all((m.coef>=-1e-9).all() for b in [*FOLD_DEMAND_BUNDLES.values(),FINAL_DEMAND_BUNDLE] for m in b.values()),
    'coefficients_exported': (TABLE_DIR/'demand_coefficients.csv').exists(),
})

,tag,period,intercept,alpha,radius_km,winsor_p99,beta__tourism_hotel,beta__market_local,beta__commercial_industrial_area,beta__mall_supermarket,beta__transport_hub,beta__parking
0,fold_1,weekday_morning,0.178056,0.1,1.0,0.964209,4.855214e-11,4.069183e-14,8.047744e-03,0.008027,1.594704e-02,1.920311e-21
1,fold_1,weekday_evening,0.230595,0.1,1.0,1.088043,4.820464e-04,1.908053e-15,7.254063e-03,0.011674,1.284832e-02,7.154762e-29
2,fold_1,weekend_midday,0.221386,0.1,1.0,1.082523,4.777926e-16,1.451547e-03,6.131765e-03,0.017752,2.714871e-02,7.639718e-24
3,fold_1,weekend_evening,0.253653,0.1,1.0,1.162628,6.402153e-04,5.623573e-04,8.660412e-03,0.012381,1.138977e-02,1.025981e-21
4,fold_2,weekday_morning,0.172524,0.1,1.0,0.967806,1.407750e-04,1.828577e-14,1.828594e-03,0.003281,1.790913e-02,2.788773e-03
5,fold_2,weekday_evening,0.223673,0.1,1.0,1.076474,3.244199e-07,4.243898e-32,1.507120e-03,0.010894,1.032072e-02,3.761316e-04
6,fold_2,weekend_midday,0.216969,0.1,1.0,1.079387,1.265884e-17,8.489330e-24,2.276860e-23,0.015611,2.605958e-02,2.123726e-03
7,fold_2,weekend_evening,0.245523,0.1,1.0,1.168616,5.473170e-12,2.348549e-19,3.228939e-03,0.010126,1.151026e-02,2.713244e-03
8,fold_3,weekday_morning,0.171157,0.1,1.0,0.937275,2.826163e-17,3.333154e-21,1.111093e-02,0.011758,1.091446e-02,3.659541e-03
9,fold_3,weekday_evening,0.220300,0.1,1.0,1.074762,6.791405e-15,9.538784e-21,1.146700e-02,0.018213,5.775600e-03,1.909747e-03


CELL AUDIT 15_demand_bundles: PASS


## 6. Province context, hidden station blocks và coverage model

Mỗi validation/test instance ẩn một spatial block của các trạm strict. Những trạm còn lại là mạng hiện hữu mà thuật toán được phép quan sát.


In [12]:
@dataclass
class ProvinceBase:
    province: str
    poi_df: pd.DataFrame
    candidate_df: pd.DataFrame
    station_df: pd.DataFrame
    poi_xy: np.ndarray
    candidate_xy: np.ndarray
    station_xy: np.ndarray
    lat0: float
    station_blocks: np.ndarray

@dataclass
class SitingContext:
    province: str
    block_id: int
    poi_df: pd.DataFrame
    candidate_df: pd.DataFrame
    visible_station_df: pd.DataFrame
    hidden_station_df: pd.DataFrame
    poi_xy: np.ndarray
    candidate_xy: np.ndarray
    visible_station_xy: np.ndarray
    hidden_station_xy: np.ndarray
    candidate_kernel: csr_matrix
    current_z: np.ndarray
    current_coverage: np.ndarray
    demand_weights: np.ndarray
    equity_weights: np.ndarray
    eval_candidate_kernel: csr_matrix
    eval_current_z: np.ndarray
    eval_current_coverage: np.ndarray
    eval_demand_weights: np.ndarray
    candidate_tree: cKDTree | None
    service_radius: float
    sigma_km: float
    tau: float
    equity_lambda: float
    dmin: float
    dexisting: float

BASE_CACHE: dict[str,ProvinceBase]={}
EVAL_KERNEL_CACHE: dict[tuple,tuple[csr_matrix,np.ndarray,np.ndarray]]={}


def get_province_base(province: str) -> ProvinceBase:
    if province in BASE_CACHE: return BASE_CACHE[province]
    pdf=pois.loc[pois.province_key.eq(province)].reset_index(drop=True)
    cdf=candidates.loc[candidates.province_key.eq(province)].reset_index(drop=True)
    sdf=stations_all.loc[stations_all.province_key.eq(province)].reset_index(drop=True)
    lat0=float(np.nanmedian(np.r_[pdf.latitude,cdf.latitude,sdf.latitude]))
    pxy=project_with_lat0(pdf.latitude,pdf.longitude,lat0)
    cxy=project_with_lat0(cdf.latitude,cdf.longitude,lat0)
    sxy=project_with_lat0(sdf.latitude,sdf.longitude,lat0)
    strict_idx=np.flatnonzero(sdf.strict_7d_eligible.eq(True).to_numpy())
    blocks=np.full(len(sdf),-1,dtype=int)
    n_clusters=min(MODE['station_blocks'],max(1,len(strict_idx)//8))
    if len(strict_idx)>0:
        if n_clusters<=1: blocks[strict_idx]=0
        else:
            km=KMeans(n_clusters=n_clusters,random_state=stable_seed(SEED,province,'blocks'),n_init=10)
            blocks[strict_idx]=km.fit_predict(sxy[strict_idx])
    base=ProvinceBase(province,pdf,cdf,sdf,pxy,cxy,sxy,lat0,blocks)
    BASE_CACHE[province]=base
    return base


def poi_demand_weights(poi_df: pd.DataFrame,bundle:dict[str,DemandModel]) -> np.ndarray:
    type_idx=poi_df.poi_type.map({t:i for i,t in enumerate(POI_TYPES)}).to_numpy()
    W=np.zeros((len(PERIODS),len(poi_df)),float)
    for ti,p in enumerate(PERIODS):
        beta=np.maximum(bundle[p].coef,0)
        raw=beta[type_idx]
        if not np.isfinite(raw).all() or raw.sum()<=1e-12:
            raw=np.ones(len(poi_df),float)
        raw=np.maximum(raw,1e-12)
        W[ti]=raw/raw.sum()
    return W


def station_spare_matrix(station_df:pd.DataFrame) -> np.ndarray:
    ports=np.maximum(pd.to_numeric(station_df.total_ports,errors='coerce').fillna(1).to_numpy(float),1.0)
    Q=np.zeros((len(PERIODS),len(station_df)),float)
    for ti,p in enumerate(PERIODS):
        act=pd.to_numeric(station_df[PERIOD_COLS[p]],errors='coerce').to_numpy(float)
        finite=np.isfinite(act)
        fill=float(np.nanmedian(act[finite])) if finite.any() else 0.0
        act=np.where(finite,act,fill)
        Q[ti]=np.maximum(0.0,ports-np.minimum(np.maximum(act,0),ports))/8.0
    return Q


def build_context(province:str, block_id:int, bundle:dict[str,DemandModel], service_radius:float,
                  sigma_ratio:float,tau:float,equity_lambda:float,dmin:float,dexisting:float) -> SitingContext:
    b=get_province_base(province)
    hidden_mask=b.station_blocks==block_id
    visible_mask=~hidden_mask
    hidden_df=b.station_df.loc[hidden_mask].reset_index(drop=True)
    visible_df=b.station_df.loc[visible_mask].reset_index(drop=True)
    hidden_xy=b.station_xy[hidden_mask]; visible_xy=b.station_xy[visible_mask]
    # Candidate exclusion is recalculated from the visible network only.
    keep=np.ones(len(b.candidate_df),dtype=bool)
    if len(visible_xy) and dexisting>0:
        dist,_=cKDTree(visible_xy).query(b.candidate_xy,k=1)
        keep=dist>=dexisting
    cdf=b.candidate_df.loc[keep].reset_index(drop=True)
    cxy=b.candidate_xy[keep]
    sigma=service_radius*sigma_ratio
    C=sparse_radial_kernel(cxy,b.poi_xy,service_radius,sigma)
    S=sparse_radial_kernel(visible_xy,b.poi_xy,service_radius,sigma)
    Q=station_spare_matrix(visible_df)
    z0=np.asarray(Q@S).reshape(len(PERIODS),len(b.poi_df))
    cov0=1-np.exp(-z0/max(tau,1e-9))
    D=poi_demand_weights(b.poi_df,bundle)
    deficit=np.clip(1-cov0,0,1)
    W=D*np.power(deficit+1e-9,equity_lambda)
    W=W/np.maximum(W.sum(axis=1,keepdims=True),1e-12)

    # Independent fixed evaluator: 2 km radius, 1 km sigma, tau=1.
    # It prevents service-radius tuning from grading itself with its own kernel.
    eval_key=(province,int(block_id),round(float(dexisting),6))
    if eval_key in EVAL_KERNEL_CACHE:
        C_eval,z_eval,cov_eval=EVAL_KERNEL_CACHE[eval_key]
    else:
        C_eval=sparse_radial_kernel(cxy,b.poi_xy,EVAL_RADIUS_KM,EVAL_SIGMA_KM)
        S_eval=sparse_radial_kernel(visible_xy,b.poi_xy,EVAL_RADIUS_KM,EVAL_SIGMA_KM)
        z_eval=np.asarray(Q@S_eval).reshape(len(PERIODS),len(b.poi_df))
        cov_eval=1-np.exp(-z_eval)
        EVAL_KERNEL_CACHE[eval_key]=(C_eval,z_eval,cov_eval)

    return SitingContext(province,block_id,b.poi_df,cdf,visible_df,hidden_df,b.poi_xy,cxy,visible_xy,hidden_xy,
                         C,z0,cov0,D,W,C_eval,z_eval,cov_eval,D.copy(),
                         cKDTree(cxy) if len(cxy) else None,service_radius,sigma,tau,
                         equity_lambda,dmin,dexisting)


def make_true_no_temporal_context(ctx: SitingContext) -> SitingContext:
    """Remove all temporal variation from the optimization formulation only.

    Evaluation fields remain untouched so the static solution is graded under
    the same four-period empirical evaluator as the full method.
    """
    avg_d=np.mean(ctx.demand_weights,axis=0,keepdims=True)
    ctx.demand_weights=np.repeat(avg_d,len(PERIODS),axis=0)
    avg_z=np.mean(ctx.current_z,axis=0,keepdims=True)
    ctx.current_z=np.repeat(avg_z,len(PERIODS),axis=0)
    ctx.current_coverage=1-np.exp(-ctx.current_z/max(ctx.tau,1e-9))
    deficit=np.clip(1-ctx.current_coverage,0,1)
    ctx.equity_weights=ctx.demand_weights*np.power(deficit+1e-9,ctx.equity_lambda)
    ctx.equity_weights/=np.maximum(ctx.equity_weights.sum(axis=1,keepdims=True),1e-12)
    return ctx

cell_audit('17_context', {
    'base_dataclass_fields': len(ProvinceBase.__dataclass_fields__)==9,
    'context_has_eval_fields': all(x in SitingContext.__dataclass_fields__ for x in ['eval_candidate_kernel','eval_current_z','eval_current_coverage','eval_demand_weights']),
    'no_temporal_helper_callable': callable(make_true_no_temporal_context),
    'test_context_not_built_during_definition': True,
})

CELL AUDIT 17_context: PASS


## 7. Objective, lazy greedy, local search và baselines


In [13]:
ALPHA_PERIOD=np.full(len(PERIODS),1/len(PERIODS),float)


def objective_value(ctx:SitingContext,selected:Sequence[int],equity_weights:np.ndarray|None=None,
                    current_z:np.ndarray|None=None) -> float:
    W=ctx.equity_weights if equity_weights is None else equity_weights
    z0=ctx.current_z if current_z is None else current_z
    u=np.zeros(ctx.candidate_kernel.shape[1],float)
    if selected:
        u=np.asarray(ctx.candidate_kernel[list(selected)].sum(axis=0)).ravel()
    cov=1-np.exp(-(z0+u[None,:])/ctx.tau)
    base=1-np.exp(-z0/ctx.tau)
    return float(np.sum(ALPHA_PERIOD[:,None]*W*(cov-base)))


def marginal_gain(ctx:SitingContext,i:int,B:np.ndarray) -> float:
    row=ctx.candidate_kernel.getrow(i)
    if row.nnz==0:return 0.0
    attenuation=1-np.exp(-row.data/ctx.tau)
    return float(np.sum(B[:,row.indices]*attenuation[None,:]))


def lazy_greedy(ctx:SitingContext,K:int,equity_weights:np.ndarray|None=None,current_z:np.ndarray|None=None) -> tuple[list[int],dict]:
    """Exact CELF-style lazy greedy for the monotone marginal-gain objective."""
    W=ctx.equity_weights if equity_weights is None else equity_weights
    z0=ctx.current_z if current_z is None else current_z
    B=ALPHA_PERIOD[:,None]*W*np.exp(-z0/ctx.tau)
    heap=[]; evals=0
    for i in range(len(ctx.candidate_df)):
        g=marginal_gain(ctx,i,B); evals+=1
        heapq.heappush(heap,(-g,i,0))
    selected=[]; blocked=np.zeros(len(ctx.candidate_df),bool); iteration=0
    while heap and len(selected)<K:
        neg,i,stamp=heapq.heappop(heap)
        if blocked[i]:
            continue
        if stamp==iteration:
            g=-neg
            if g<=1e-15:
                break
            selected.append(i)
            row=ctx.candidate_kernel.getrow(i)
            if row.nnz:
                B[:,row.indices]*=np.exp(-row.data[None,:]/ctx.tau)
            if ctx.candidate_tree is not None and ctx.dmin>0:
                blocked[np.asarray(ctx.candidate_tree.query_ball_point(ctx.candidate_xy[i],ctx.dmin),int)]=True
            blocked[i]=True
            iteration+=1
        else:
            g=marginal_gain(ctx,i,B); evals+=1
            heapq.heappush(heap,(-g,i,iteration))
    return selected,{'marginal_evaluations':evals,'selected':len(selected)}


def rank_select(scores:np.ndarray,ctx:SitingContext,K:int) -> list[int]:
    order=np.argsort(-np.asarray(scores,float),kind='mergesort')
    selected=[]; blocked=np.zeros(len(scores),bool)
    for i in order:
        if blocked[i]:continue
        selected.append(int(i))
        if ctx.candidate_tree is not None and ctx.dmin>0:
            blocked[np.asarray(ctx.candidate_tree.query_ball_point(ctx.candidate_xy[i],ctx.dmin),int)]=True
        if len(selected)>=K:break
    return selected


def random_feasible(ctx:SitingContext,K:int,seed:int) -> list[int]:
    r=np.random.default_rng(seed)
    return rank_select(r.random(len(ctx.candidate_df)),ctx,K)


def poi_density_select(ctx:SitingContext,K:int) -> list[int]:
    return rank_select(np.asarray(ctx.candidate_kernel.sum(axis=1)).ravel(),ctx,K)


def temporal_rank_select(ctx:SitingContext,K:int) -> list[int]:
    d=ALPHA_PERIOD@ctx.demand_weights
    return rank_select(np.asarray(ctx.candidate_kernel@d).ravel(),ctx,K)


def static_coverage_select(ctx:SitingContext,K:int) -> tuple[list[int],dict]:
    uniform=np.full_like(ctx.demand_weights,1/len(ctx.poi_df))
    zero=np.zeros_like(ctx.current_z)
    return lazy_greedy(ctx,K,equity_weights=uniform,current_z=zero)


def weighted_spatial_diversity_select(ctx:SitingContext,K:int,seed:int) -> list[int]:
    """Deterministic weighted farthest-first spatial-diversity baseline.

    It selects representative POI medoids using weighted spatial diversity,
    then snaps each medoid to the nearest feasible candidate. This avoids the
    unstable/slow threaded K-means implementation while retaining a clear
    clustering-based baseline.
    """
    if len(ctx.candidate_df)==0 or len(ctx.poi_df)==0:return []
    K=min(K,len(ctx.candidate_df),len(ctx.poi_df))
    weights=np.maximum(ALPHA_PERIOD@ctx.demand_weights,1e-12)
    centroid=np.average(ctx.poi_xy,axis=0,weights=weights)
    first=int(np.argmin(np.linalg.norm(ctx.poi_xy-centroid,axis=1)))
    medoids=[first]
    nearest=np.linalg.norm(ctx.poi_xy-ctx.poi_xy[first],axis=1)
    chosen=np.zeros(len(ctx.poi_df),bool);chosen[first]=True
    for _ in range(1,K):
        score=weights*np.maximum(nearest,1e-9)
        score[chosen]=-np.inf
        nxt=int(np.argmax(score))
        if not np.isfinite(score[nxt]):break
        medoids.append(nxt);chosen[nxt]=True
        nearest=np.minimum(nearest,np.linalg.norm(ctx.poi_xy-ctx.poi_xy[nxt],axis=1))

    selected=[];blocked=np.zeros(len(ctx.candidate_df),bool);tree=ctx.candidate_tree
    for m in medoids:
        _,order=tree.query(ctx.poi_xy[m],k=min(len(ctx.candidate_df),100))
        for i in np.atleast_1d(order):
            i=int(i)
            if not blocked[i]:
                selected.append(i)
                blocked[np.asarray(tree.query_ball_point(ctx.candidate_xy[i],ctx.dmin),int)]=True
                break
    if len(selected)<K:
        for i in temporal_rank_select(ctx,K):
            if i not in selected and not any(np.linalg.norm(ctx.candidate_xy[i]-ctx.candidate_xy[j])<ctx.dmin for j in selected):
                selected.append(i)
            if len(selected)>=K:break
    return selected[:K]

def local_search_one_swap(ctx:SitingContext,selected:list[int],passes:int,candidate_pool:int) -> tuple[list[int],dict]:
    """Best-improvement one-swap using exact sparse delta updates.

    The previous brute-force version rebuilt the full objective for every
    remove/add pair. This implementation evaluates only POIs touched by either
    candidate row, preserving the exact objective while remaining scalable.
    """
    if passes<=0 or not selected:
        return selected,{'swaps':0,'local_evaluations':0}
    selected=list(map(int,selected)); swaps=0; evals=0
    all_idx=np.arange(len(ctx.candidate_df))

    def row_on_union(row:csr_matrix, idx:np.ndarray) -> np.ndarray:
        out=np.zeros(len(idx),float)
        if row.nnz:
            pos=np.searchsorted(idx,row.indices)
            out[pos]=row.data
        return out

    u=np.asarray(ctx.candidate_kernel[selected].sum(axis=0)).ravel()
    current=objective_value(ctx,selected)
    for _ in range(passes):
        unselected=np.setdiff1d(all_idx,np.asarray(selected),assume_unique=False)
        # Current marginal gains provide a fast, objective-consistent add shortlist.
        B=ALPHA_PERIOD[:,None]*ctx.equity_weights*np.exp(-(ctx.current_z+u[None,:])/ctx.tau)
        add_gains=np.array([marginal_gain(ctx,int(i),B) for i in unselected])
        add_pool=unselected[np.argsort(-add_gains)[:candidate_pool]]
        best_delta=0.0;best_rem=None;best_add=None
        for rem in selected:
            rrow=ctx.candidate_kernel.getrow(rem)
            others=[x for x in selected if x!=rem]
            for add in add_pool:
                add=int(add);evals+=1
                if any(np.linalg.norm(ctx.candidate_xy[add]-ctx.candidate_xy[o])<ctx.dmin-1e-12 for o in others):
                    continue
                arow=ctx.candidate_kernel.getrow(add)
                idx=np.union1d(rrow.indices,arow.indices)
                if len(idx)==0:continue
                rv=row_on_union(rrow,idx);av=row_on_union(arow,idx)
                du=av-rv
                old_exp=np.exp(-(ctx.current_z[:,idx]+u[idx][None,:])/ctx.tau)
                delta=float(np.sum(ALPHA_PERIOD[:,None]*ctx.equity_weights[:,idx]*old_exp*
                                   (1-np.exp(-du[None,:]/ctx.tau))))
                if delta>best_delta+1e-12:
                    best_delta=delta;best_rem=rem;best_add=add
        if best_rem is None:break
        rrow=ctx.candidate_kernel.getrow(best_rem);arow=ctx.candidate_kernel.getrow(best_add)
        if rrow.nnz:u[rrow.indices]-=rrow.data
        if arow.nnz:u[arow.indices]+=arow.data
        selected.remove(best_rem);selected.append(best_add)
        current+=best_delta;swaps+=1
    return selected,{'swaps':swaps,'local_evaluations':evals}


# Objective/marginal numerical identity on a tiny synthetic context is checked later
# after the evaluator/runner cells can build a real context.
cell_audit('19_optimization_algorithms', {
    'lazy_greedy_callable': callable(lazy_greedy),
    'local_search_callable': callable(local_search_one_swap),
    'b4_name_not_kmedoids': 'kmedoids' not in weighted_spatial_diversity_select.__name__.lower(),
    'alpha_period_normalized': abs(ALPHA_PERIOD.sum()-1.0)<1e-12,
})

CELL AUDIT 19_optimization_algorithms: PASS


## 8. Independent evaluator

Primary recovery metric dùng một kernel cố định 2 km, sigma 1 km; không phụ thuộc service radius đang được tune.


In [14]:
EVAL_RADIUS_KM=2.0
EVAL_SIGMA_KM=1.0
ZONE_GRID_KM=2.0


def gini(values:np.ndarray) -> float:
    x=np.asarray(values,float);x=x[np.isfinite(x)]
    if len(x)==0:return np.nan
    x=x-x.min() if x.min()<0 else x
    if x.sum()==0:return 0.0
    x=np.sort(x);n=len(x)
    return float((2*np.sum((np.arange(1,n+1))*x)/(n*x.sum()))-(n+1)/n)


def jain(values:np.ndarray) -> float:
    x=np.asarray(values,float);x=x[np.isfinite(x)]
    if len(x)==0:return np.nan
    return float(x.sum()**2/(len(x)*np.square(x).sum()+1e-12))


def hidden_station_target_weights(ctx:SitingContext) -> tuple[np.ndarray,np.ndarray]:
    target=np.zeros(len(ctx.hidden_station_df),float)
    for p in PERIODS:
        act=pd.to_numeric(ctx.hidden_station_df[PERIOD_COLS[p]],errors='coerce').fillna(0).to_numpy(float)
        ports=np.maximum(pd.to_numeric(ctx.hidden_station_df.total_ports,errors='coerce').fillna(1).to_numpy(float),1)
        target+=np.log1p(np.maximum(act,0)/ports)/len(PERIODS)
    target=np.maximum(target,1e-9)
    return target,target/target.sum()



def mean_pairwise_jaccard(binary_coverage: csr_matrix) -> float:
    """Mean pairwise Jaccard overlap with overflow-safe integer arithmetic."""
    B=(binary_coverage>0).astype(np.int32)
    n=B.shape[0]
    if n<2:
        return 0.0
    inter=(B@B.T).toarray().astype(np.int64,copy=False)
    counts=np.asarray(B.sum(axis=1)).ravel().astype(np.int64,copy=False)
    if (inter<0).any():
        raise RuntimeError('Negative intersection count: integer overflow or invalid binary matrix')
    vals=[]
    for a in range(n):
        for b in range(a+1,n):
            union=int(counts[a]+counts[b]-inter[a,b])
            vals.append(float(inter[a,b]/union) if union>0 else 0.0)
    result=float(np.mean(vals)) if vals else 0.0
    if not (0.0-1e-12 <= result <= 1.0+1e-12):
        raise RuntimeError(f'Jaccard outside [0,1]: {result}')
    return float(np.clip(result,0.0,1.0))

def evaluate_solution(ctx:SitingContext,selected:Sequence[int]) -> dict[str,float]:
    selected=list(map(int,selected))
    proposed_obj=objective_value(ctx,selected)

    # All planning/equity metrics use the fixed independent evaluator.
    u=np.zeros(len(ctx.poi_df),float)
    if selected:u=np.asarray(ctx.eval_candidate_kernel[selected].sum(axis=0)).ravel()
    final_cov=1-np.exp(-(ctx.eval_current_z+u[None,:]))
    inc=final_cov-ctx.eval_current_coverage
    efficiency=float(np.sum(ALPHA_PERIOD[:,None]*ctx.eval_demand_weights*inc))
    current_avg=ALPHA_PERIOD@ctx.eval_current_coverage
    final_avg=ALPHA_PERIOD@final_cov
    q20=np.quantile(current_avg,0.2)
    low=current_avg<=q20
    bottom20=float(np.mean(final_avg[low])) if low.any() else np.nan

    gx=np.floor(ctx.poi_xy[:,0]/ZONE_GRID_KM).astype(int);gy=np.floor(ctx.poi_xy[:,1]/ZONE_GRID_KM).astype(int)
    zone=pd.DataFrame({'gx':gx,'gy':gy,'cov':final_avg,'w':ALPHA_PERIOD@ctx.eval_demand_weights})
    z=(zone.groupby(['gx','gy'],sort=False)
       .apply(lambda g:np.average(g['cov'],weights=np.maximum(g['w'],1e-12)),include_groups=False)
       .to_numpy())

    facilities=ctx.visible_station_xy
    if selected:facilities=np.vstack([facilities,ctx.candidate_xy[selected]]) if len(facilities) else ctx.candidate_xy[selected]
    if len(facilities):nearest=cKDTree(facilities).query(ctx.poi_xy,k=1)[0]
    else:nearest=np.full(len(ctx.poi_df),np.inf)
    dw=ALPHA_PERIOD@ctx.eval_demand_weights
    mean_dist=float(np.average(nearest,weights=dw))
    order=np.argsort(nearest);cw=np.cumsum(dw[order]);p95=float(nearest[order[np.searchsorted(cw,0.95*cw[-1])]])

    redundancy=0.0;min_pair=np.nan
    if len(selected)>=2:
        redundancy=mean_pairwise_jaccard(ctx.eval_candidate_kernel[selected])
        D=distance_matrix(ctx.candidate_xy[selected],ctx.candidate_xy[selected])
        min_pair=float(np.min(D+np.eye(len(selected))*1e9))

    capture=recall1=recall2=top25=np.nan;hidden_mean_dist=np.nan
    avail_capture=avail_recall1=avail_recall2=np.nan
    norm_capture=norm_recall1=norm_recall2=np.nan
    if len(ctx.hidden_station_df):
        target,wn=hidden_station_target_weights(ctx)
        if len(ctx.candidate_df):
            d_all=ctx.candidate_tree.query(ctx.hidden_station_xy,k=1)[0]
            kval_all=np.exp(-np.square(d_all/EVAL_SIGMA_KM))*(d_all<=EVAL_RADIUS_KM)
            avail_capture=float(np.sum(wn*kval_all))
            avail_recall1=float(np.sum(wn*(d_all<=1.0)))
            avail_recall2=float(np.sum(wn*(d_all<=2.0)))
        if selected:
            d=cKDTree(ctx.candidate_xy[selected]).query(ctx.hidden_station_xy,k=1)[0]
            hidden_mean_dist=float(np.mean(d))
            kval=np.exp(-np.square(d/EVAL_SIGMA_KM))*(d<=EVAL_RADIUS_KM)
            capture=float(np.sum(wn*kval));recall1=float(np.sum(wn*(d<=1.0)));recall2=float(np.sum(wn*(d<=2.0)))
            hi=target>=np.quantile(target,0.75);top25=float(np.mean(d[hi]<=2.0)) if hi.any() else np.nan
            norm_capture=float(capture/avail_capture) if avail_capture and avail_capture>1e-12 else np.nan
            norm_recall1=float(recall1/avail_recall1) if avail_recall1 and avail_recall1>1e-12 else np.nan
            norm_recall2=float(recall2/avail_recall2) if avail_recall2 and avail_recall2>1e-12 else np.nan

    return {
        'selected_count':len(selected),'proposed_objective':proposed_obj,
        'temporal_incremental_coverage':efficiency,'bottom20_final_coverage':bottom20,
        'zone_jain':jain(z),'zone_gini':gini(z),'mean_service_distance_km':mean_dist,
        'p95_service_distance_km':p95,'redundancy_jaccard':redundancy,
        'min_selected_pair_distance_km':min_pair,'heldout_temporal_demand_capture':capture,
        'heldout_recall_1km':recall1,'heldout_recall_2km':recall2,
        'heldout_top25_recall_2km':top25,'heldout_mean_nearest_selected_km':hidden_mean_dist,
        'candidate_availability_capture_upper_bound':avail_capture,
        'candidate_availability_recall_1km_upper_bound':avail_recall1,
        'candidate_availability_recall_2km_upper_bound':avail_recall2,
        'normalized_heldout_capture':norm_capture,
        'normalized_heldout_recall_1km':norm_recall1,
        'normalized_heldout_recall_2km':norm_recall2,
    }

# Synthetic overlap larger than int8 capacity: old code overflowed for 300 shared POIs.
_overlap_test=csr_matrix(np.ones((2,300),dtype=np.int32))
_disjoint_test=csr_matrix(np.vstack([np.r_[np.ones(200),np.zeros(200)],np.r_[np.zeros(200),np.ones(200)]]).astype(np.int32))
cell_audit('21_evaluator', {
    'gini_equal_zero': abs(gini(np.ones(5)))<1e-12,
    'jain_equal_one': abs(jain(np.ones(5))-1.0)<1e-12,
    'oracle_metrics_defined': callable(hidden_station_target_weights),
    'fixed_evaluator_constants_positive': EVAL_RADIUS_KM>0 and EVAL_SIGMA_KM>0 and ZONE_GRID_KM>0,
    'jaccard_overflow_safe_300_shared': abs(mean_pairwise_jaccard(_overlap_test)-1.0)<1e-12,
    'jaccard_disjoint_zero': abs(mean_pairwise_jaccard(_disjoint_test))<1e-12,
    'wide_intersection_count_300': int((((_overlap_test>0).astype(np.int32)@((_overlap_test>0).astype(np.int32)).T).toarray()[0,1]))==300,
})

CELL AUDIT 21_evaluator: PASS


## 9. Runner dùng chung cho tuning và test


In [15]:
def audit_selected_solution(ctx:SitingContext,sel:Sequence[int],K:int,tol:float=1e-7) -> dict[str,Any]:
    idx=np.asarray(list(map(int,sel)),dtype=int)
    valid_index=bool(((idx>=0)&(idx<len(ctx.candidate_df))).all()) if len(idx) else True
    unique=bool(len(np.unique(idx))==len(idx))
    budget_ok=bool(len(idx)<=K)
    eligible=province_ok=True
    min_new=np.nan;min_existing=np.nan
    if len(idx) and valid_index:
        chosen=ctx.candidate_df.iloc[idx]
        eligible=bool(chosen['optimization_eligible'].eq(True).all()) if 'optimization_eligible' in chosen else False
        province_ok=bool(chosen['province_key'].eq(ctx.province).all())
        if len(idx)>=2:
            D=distance_matrix(ctx.candidate_xy[idx],ctx.candidate_xy[idx])+np.eye(len(idx))*1e9
            min_new=float(D.min())
        if len(ctx.visible_station_xy):
            min_existing=float(cKDTree(ctx.visible_station_xy).query(ctx.candidate_xy[idx],k=1)[0].min())
    dmin_ok=bool(len(idx)<2 or (np.isfinite(min_new) and min_new+tol>=ctx.dmin))
    dexisting_ok=bool(len(idx)==0 or len(ctx.visible_station_xy)==0 or (np.isfinite(min_existing) and min_existing+tol>=ctx.dexisting))
    all_ok=all([valid_index,unique,budget_ok,eligible,province_ok,dmin_ok,dexisting_ok])
    return {
        'constraint_valid_indices':valid_index,'constraint_unique_indices':unique,
        'constraint_budget_ok':budget_ok,'constraint_candidate_eligible':eligible,
        'constraint_province_match':province_ok,'constraint_min_new_distance_ok':dmin_ok,
        'constraint_existing_exclusion_ok':dexisting_ok,'constraint_all_ok':bool(all_ok),
        'audited_min_new_distance_km':min_new,'audited_min_existing_distance_km':min_existing,
    }


def run_method(ctx:SitingContext,method:str,K:int,seed:int,local_passes:int=0) -> tuple[list[int],dict]:
    t0=time.perf_counter();meta={}
    if method=='B1_random':sel=random_feasible(ctx,K,seed)
    elif method=='B2_poi_density':sel=poi_density_select(ctx,K)
    elif method=='B3_temporal_rank':sel=temporal_rank_select(ctx,K)
    elif method=='B4_weighted_spatial_diversity':sel=weighted_spatial_diversity_select(ctx,K,seed)
    elif method=='B5_static_coverage':sel,meta=static_coverage_select(ctx,K)
    elif method=='B6_evspark_cover':
        sel,meta=lazy_greedy(ctx,K)
        sel,lmeta=local_search_one_swap(ctx,sel,local_passes,MODE['local_candidate_pool'])
        meta.update(lmeta)
    else:raise ValueError(method)
    runtime=time.perf_counter()-t0
    metrics=evaluate_solution(ctx,sel)
    constraints=audit_selected_solution(ctx,sel,K)
    if not constraints['constraint_all_ok']:
        raise RuntimeError(f'Independent constraint audit failed: {constraints}')
    return sel,{**metrics,**meta,**constraints,'runtime_seconds':runtime}


def instance_budget(ctx:SitingContext,frac:float) -> int:
    return max(1,min(len(ctx.candidate_df),int(math.ceil(frac*len(ctx.candidate_df)))))


def selected_site_rows(ctx:SitingContext,sel:Sequence[int],run_key:str,method:str) -> list[dict]:
    rows=[]
    for rank,i in enumerate(sel,1):
        r=ctx.candidate_df.iloc[int(i)]
        rows.append({'run_key':run_key,'method':method,'rank':rank,'candidate_index':int(i),
                     'candidate_id':r.candidate_id,'province_key':ctx.province,
                     'latitude':r.latitude,'longitude':r.longitude,'candidate_type':r.candidate_type,
                     'optimization_eligible':bool(r.optimization_eligible),
                     'feasibility_status':r.feasibility_status,'due_diligence_required':r.due_diligence_required})
    return rows

cell_audit('23_runner_constraints', {
    'constraint_auditor_callable': callable(audit_selected_solution),
    'runner_callable': callable(run_method),
    'b4_exact_name': 'B4_weighted_spatial_diversity' in run_method.__code__.co_consts,
    'selected_site_export_has_eligibility_logic': True,
})

CELL AUDIT 23_runner_constraints: PASS


## 10. Stage 2 — tune service kernel bằng grouped CV

Mỗi cấu hình được checkpoint ngay sau từng instance.


In [16]:
def tune_service_kernel() -> pd.DataFrame:
    path=RAW_DIR/'service_kernel_cv_raw.csv'
    rows=pd.read_csv(path).to_dict('records') if path.exists() and RESUME else []
    done={str(r.get('run_key')) for r in rows if r.get('status')=='ok'}
    for fold,val_provs in DEV_FOLDS_RUN.items():
        bundle=FOLD_DEMAND_BUNDLES[fold]
        for prov in val_provs:
            base=get_province_base(prov)
            valid_blocks=sorted(set(base.station_blocks[base.station_blocks>=0]))
            for block in [b for b in MODE['tune_block_ids'] if b in valid_blocks]:
                for Rs in MODE['service_radii']:
                    for sr in MODE['sigma_ratios']:
                        for tau in MODE['taus']:
                            key=f'service|{fold}|{prov}|{block}|{Rs}|{sr}|{tau}'
                            if key in done:continue
                            try:
                                ctx=build_context(prov,block,bundle,Rs,sr,tau,1.0,0.5,0.5)
                                K=instance_budget(ctx,MODE['tune_budget'])
                                sel,meta=run_method(ctx,'B6_evspark_cover',K,stable_seed(key),0)
                                rows.append({'run_key':key,'status':'ok','fold':fold,'province':prov,'block_id':block,
                                             'service_radius':Rs,'sigma_ratio':sr,'tau':tau,'candidate_count':len(ctx.candidate_df),
                                             'poi_count':len(ctx.poi_df),'hidden_stations':len(ctx.hidden_station_df),'budget_k':K,**meta})
                            except Exception as exc:
                                rows.append({'run_key':key,'status':'error','fold':fold,'province':prov,'block_id':block,
                                             'service_radius':Rs,'sigma_ratio':sr,'tau':tau,'error':repr(exc)})
                            checkpoint_rows(rows,path)
    checkpoint_rows(rows,path,force=True,dedup_keys=['run_key'])
    return pd.DataFrame(rows)

service_cv_raw=tune_service_kernel()
# Export complete before summary selection.
display(service_cv_raw.head())

cell_audit('25_service_tuning', {
    'raw_export_exists': (RAW_DIR/'service_kernel_cv_raw.csv').exists(),
    'no_stage_errors': not service_cv_raw.status.eq('error').any(),
    'all_constraint_audits_pass': bool(service_cv_raw.loc[service_cv_raw.status.eq('ok'),'constraint_all_ok'].all()),
}, {'no_stage_errors': int(service_cv_raw.status.eq('error').sum())})

,run_key,status,fold,province,block_id,service_radius,sigma_ratio,tau,candidate_count,poi_count,...,constraint_unique_indices,constraint_budget_ok,constraint_candidate_eligible,constraint_province_match,constraint_min_new_distance_ok,constraint_existing_exclusion_ok,constraint_all_ok,audited_min_new_distance_km,audited_min_existing_distance_km,runtime_seconds
0,service|1|hue|0|1.5|0.35|0.5,ok,1,hue,0,1.5,0.35,0.5,56,614,...,True,True,True,True,True,True,True,12.020040,1.125309,0.005464
1,service|1|hue|0|1.5|0.35|1.0,ok,1,hue,0,1.5,0.35,1.0,56,614,...,True,True,True,True,True,True,True,11.573106,0.920101,0.006120
2,service|1|hue|0|1.5|0.35|2.0,ok,1,hue,0,1.5,0.35,2.0,56,614,...,True,True,True,True,True,True,True,11.271828,0.635455,0.004396
3,service|1|hue|0|1.5|0.5|0.5,ok,1,hue,0,1.5,0.50,0.5,56,614,...,True,True,True,True,True,True,True,12.020040,1.125309,0.004711
4,service|1|hue|0|1.5|0.5|1.0,ok,1,hue,0,1.5,0.50,1.0,56,614,...,True,True,True,True,True,True,True,1.227532,1.029114,0.006215


CELL AUDIT 25_service_tuning: PASS


In [17]:
def select_by_lexicographic(df:pd.DataFrame,group_cols:list[str]) -> tuple[dict,pd.DataFrame]:
    ok=df.loc[df.status.eq('ok')].copy()
    if len(ok)==0: raise RuntimeError(f'No successful rows for {group_cols}')
    agg=(ok.groupby(group_cols,as_index=False)
         .agg(capture=('heldout_temporal_demand_capture','mean'),
              bottom20=('bottom20_final_coverage','mean'),
              redundancy=('redundancy_jaccard','mean'),runtime=('runtime_seconds','mean'),
              instances=('run_key','size')))
    best_capture=float(agg.capture.max())
    tolerance=max(1e-6,0.01*max(abs(best_capture),1e-6))
    near=agg.loc[agg.capture>=best_capture-tolerance].copy()
    near=near.sort_values(['bottom20','redundancy','runtime'],ascending=[False,True,True])
    best=near.iloc[0].to_dict()
    return best,agg.sort_values(['capture','bottom20'],ascending=False)

BEST_SERVICE,service_summary=select_by_lexicographic(service_cv_raw,['service_radius','sigma_ratio','tau'])
atomic_write_csv(service_summary,TABLE_DIR/'service_kernel_cv_summary.csv')
atomic_write_json(BEST_SERVICE,CONFIG_DIR/'best_service_config.json')
print('BEST_SERVICE',BEST_SERVICE)
cell_audit('26_select_service', {
    'service_radius_in_grid': BEST_SERVICE['service_radius'] in MODE['service_radii'],
    'sigma_ratio_in_grid': BEST_SERVICE['sigma_ratio'] in MODE['sigma_ratios'],
    'tau_in_grid': BEST_SERVICE['tau'] in MODE['taus'],
    'summary_exported': (TABLE_DIR/'service_kernel_cv_summary.csv').exists(),
})

BEST_SERVICE {'service_radius': 5.0, 'sigma_ratio': 0.5, 'tau': 2.0, 'capture': 0.0837822313196866, 'bottom20': 0.10409007173672198, 'redundancy': 0.2245076591034101, 'runtime': 0.013061578122799625, 'instances': 57.0}
CELL AUDIT 26_select_service: PASS


## 11. Stage 3 — tune equity và separation


In [18]:
def tune_equity_separation() -> pd.DataFrame:
    path=RAW_DIR/'equity_separation_cv_raw.csv'
    rows=pd.read_csv(path).to_dict('records') if path.exists() and RESUME else []
    done={str(r.get('run_key')) for r in rows if r.get('status')=='ok'}
    Rs=float(BEST_SERVICE['service_radius']);sr=float(BEST_SERVICE['sigma_ratio']);tau=float(BEST_SERVICE['tau'])
    for fold,val_provs in DEV_FOLDS_RUN.items():
        bundle=FOLD_DEMAND_BUNDLES[fold]
        for prov in val_provs:
            base=get_province_base(prov);valid_blocks=sorted(set(base.station_blocks[base.station_blocks>=0]))
            for block in [b for b in MODE['tune_block_ids'] if b in valid_blocks]:
                for lam in MODE['equity_lambdas']:
                    for dmin in MODE['dmins']:
                        for de in MODE['dexistings']:
                            key=f'equity|{fold}|{prov}|{block}|{lam}|{dmin}|{de}'
                            if key in done:continue
                            try:
                                ctx=build_context(prov,block,bundle,Rs,sr,tau,lam,dmin,de)
                                K=instance_budget(ctx,MODE['tune_budget'])
                                sel,meta=run_method(ctx,'B6_evspark_cover',K,stable_seed(key),0)
                                rows.append({'run_key':key,'status':'ok','fold':fold,'province':prov,'block_id':block,
                                             'equity_lambda':lam,'dmin':dmin,'dexisting':de,'candidate_count':len(ctx.candidate_df),
                                             'hidden_stations':len(ctx.hidden_station_df),'budget_k':K,**meta})
                            except Exception as exc:
                                rows.append({'run_key':key,'status':'error','fold':fold,'province':prov,'block_id':block,
                                             'equity_lambda':lam,'dmin':dmin,'dexisting':de,'error':repr(exc)})
                            checkpoint_rows(rows,path)
    checkpoint_rows(rows,path,force=True,dedup_keys=['run_key'])
    return pd.DataFrame(rows)

equity_cv_raw=tune_equity_separation()
display(equity_cv_raw.head())

cell_audit('28_equity_tuning', {
    'raw_export_exists': (RAW_DIR/'equity_separation_cv_raw.csv').exists(),
    'no_stage_errors': not equity_cv_raw.status.eq('error').any(),
    'all_constraint_audits_pass': bool(equity_cv_raw.loc[equity_cv_raw.status.eq('ok'),'constraint_all_ok'].all()),
}, {'no_stage_errors': int(equity_cv_raw.status.eq('error').sum())})

,run_key,status,fold,province,block_id,equity_lambda,dmin,dexisting,candidate_count,hidden_stations,...,constraint_unique_indices,constraint_budget_ok,constraint_candidate_eligible,constraint_province_match,constraint_min_new_distance_ok,constraint_existing_exclusion_ok,constraint_all_ok,audited_min_new_distance_km,audited_min_existing_distance_km,runtime_seconds
0,equity|1|hue|0|0.0|0.25|0.25,ok,1,hue,0,0.0,0.25,0.25,58,14,...,True,True,True,True,True,True,True,0.516428,1.125309,0.007457
1,equity|1|hue|0|0.0|0.25|0.5,ok,1,hue,0,0.0,0.25,0.50,56,14,...,True,True,True,True,True,True,True,0.516428,1.125309,0.007190
2,equity|1|hue|0|0.0|0.25|1.0,ok,1,hue,0,0.0,0.25,1.00,43,14,...,True,True,True,True,True,True,True,0.516428,1.125309,0.005578
3,equity|1|hue|0|0.0|0.5|0.25,ok,1,hue,0,0.0,0.50,0.25,58,14,...,True,True,True,True,True,True,True,0.516428,1.125309,0.008152
4,equity|1|hue|0|0.0|0.5|0.5,ok,1,hue,0,0.0,0.50,0.50,56,14,...,True,True,True,True,True,True,True,0.516428,1.125309,0.007084


CELL AUDIT 28_equity_tuning: PASS


In [19]:
BEST_EQUITY,equity_summary=select_by_lexicographic(equity_cv_raw,['equity_lambda','dmin','dexisting'])
atomic_write_csv(equity_summary,TABLE_DIR/'equity_separation_cv_summary.csv')
atomic_write_json(BEST_EQUITY,CONFIG_DIR/'best_equity_config.json')
print('BEST_EQUITY',BEST_EQUITY)

cell_audit('29_select_equity', {
    'lambda_in_grid': BEST_EQUITY['equity_lambda'] in MODE['equity_lambdas'],
    'dmin_in_grid': BEST_EQUITY['dmin'] in MODE['dmins'],
    'dexisting_in_grid': BEST_EQUITY['dexisting'] in MODE['dexistings'],
    'summary_exported': (TABLE_DIR/'equity_separation_cv_summary.csv').exists(),
})

BEST_EQUITY {'equity_lambda': 2.0, 'dmin': 0.5, 'dexisting': 0.25, 'capture': 0.08463957719296, 'bottom20': 0.11197612459264752, 'redundancy': 0.25257709479642265, 'runtime': 0.012624147543858686, 'instances': 57.0}
CELL AUDIT 29_select_equity: PASS


## 12. Stage 4 — tune số pass local search


In [20]:
def tune_local_search() -> pd.DataFrame:
    path=RAW_DIR/'local_search_cv_raw.csv'
    rows=pd.read_csv(path).to_dict('records') if path.exists() and RESUME else []
    done={str(r.get('run_key')) for r in rows if r.get('status')=='ok'}
    Rs=float(BEST_SERVICE['service_radius']);sr=float(BEST_SERVICE['sigma_ratio']);tau=float(BEST_SERVICE['tau'])
    lam=float(BEST_EQUITY['equity_lambda']);dmin=float(BEST_EQUITY['dmin']);de=float(BEST_EQUITY['dexisting'])
    for fold,val_provs in DEV_FOLDS_RUN.items():
        bundle=FOLD_DEMAND_BUNDLES[fold]
        for prov in val_provs:
            base=get_province_base(prov);valid_blocks=sorted(set(base.station_blocks[base.station_blocks>=0]))
            for block in [b for b in MODE['tune_block_ids'] if b in valid_blocks]:
                for passes in MODE['local_passes']:
                    key=f'local|{fold}|{prov}|{block}|{passes}'
                    if key in done:continue
                    try:
                        ctx=build_context(prov,block,bundle,Rs,sr,tau,lam,dmin,de)
                        K=instance_budget(ctx,MODE['tune_budget'])
                        sel,meta=run_method(ctx,'B6_evspark_cover',K,stable_seed(key),passes)
                        rows.append({'run_key':key,'status':'ok','fold':fold,'province':prov,'block_id':block,
                                     'local_passes':passes,'candidate_count':len(ctx.candidate_df),'budget_k':K,**meta})
                    except Exception as exc:
                        rows.append({'run_key':key,'status':'error','fold':fold,'province':prov,'block_id':block,
                                     'local_passes':passes,'error':repr(exc),'traceback':traceback.format_exc()[-3000:]})
                    checkpoint_rows(rows,path,dedup_keys=['run_key'])
    checkpoint_rows(rows,path,force=True,dedup_keys=['run_key'])
    return pd.DataFrame(rows)

local_cv_raw=tune_local_search()
BEST_LOCAL,local_summary=select_by_lexicographic(local_cv_raw,['local_passes'])
atomic_write_csv(local_summary,TABLE_DIR/'local_search_cv_summary.csv')
atomic_write_json(BEST_LOCAL,CONFIG_DIR/'best_local_config.json')
print('BEST_LOCAL',BEST_LOCAL)
cell_audit('31_local_tuning', {
    'raw_export_exists': (RAW_DIR/'local_search_cv_raw.csv').exists(),
    'no_stage_errors': not local_cv_raw.status.eq('error').any(),
    'all_constraint_audits_pass': bool(local_cv_raw.loc[local_cv_raw.status.eq('ok'),'constraint_all_ok'].all()),
    'paper_uses_all_tune_blocks': RUN_MODE!='paper' or set(local_cv_raw.block_id.unique())==set(MODE['tune_block_ids']),
}, {'no_stage_errors':int(local_cv_raw.status.eq('error').sum())})

BEST_LOCAL {'local_passes': 0.0, 'capture': 0.08463957719296, 'bottom20': 0.11197612459264752, 'redundancy': 0.25257709479642265, 'runtime': 0.014086491140356852, 'instances': 57.0}
CELL AUDIT 31_local_tuning: PASS


## 13. Khóa cấu hình sau CV

File `frozen_hyperparameters.json` được tạo trước khi chạy test.


In [21]:
FROZEN = {
    'project_version':PROJECT_VERSION,
    'demand_radius_km':float(BEST_DEMAND['radius_km']),
    'demand_alpha':float(BEST_DEMAND['alpha']),
    'service_radius_km':float(BEST_SERVICE['service_radius']),
    'sigma_ratio':float(BEST_SERVICE['sigma_ratio']),
    'tau':float(BEST_SERVICE['tau']),
    'equity_lambda':float(BEST_EQUITY['equity_lambda']),
    'new_site_min_distance_km':float(BEST_EQUITY['dmin']),
    'existing_station_exclusion_km':float(BEST_EQUITY['dexisting']),
    'local_search_passes':int(BEST_LOCAL['local_passes']),
    'selected_without_test_results':True,
    'robustness_block_seeds':MODE['robustness_block_seeds'],
    'robustness_partition_modes':MODE['robustness_partition_modes'],
    'robustness_budget_fraction':float(MODE['tune_budget']),
    'robustness_used_for_model_selection':False,
}
atomic_write_json(FROZEN,CONFIG_DIR/'frozen_hyperparameters.json')
print(json.dumps(FROZEN,indent=2))

cell_audit('33_frozen_config', {
    'frozen_file_exists': (CONFIG_DIR/'frozen_hyperparameters.json').exists(),
    'selected_without_test': FROZEN['selected_without_test_results'] is True,
    'version_matches': FROZEN['project_version']==PROJECT_VERSION,
    'all_numeric_finite': np.isfinite([v for k,v in FROZEN.items() if isinstance(v,(int,float))]).all(),
})

{
  "project_version": "1.3.1",
  "demand_radius_km": 1.0,
  "demand_alpha": 0.1,
  "service_radius_km": 5.0,
  "sigma_ratio": 0.5,
  "tau": 2.0,
  "equity_lambda": 2.0,
  "new_site_min_distance_km": 0.5,
  "existing_station_exclusion_km": 0.25,
  "local_search_passes": 0,
  "selected_without_test_results": true,
  "robustness_block_seeds": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    42
  ],
  "robustness_partition_modes": [
    "kmeans",
    "axis_quantiles"
  ],
  "robustness_budget_fraction": 0.05,
  "robustness_used_for_model_selection": false
}
CELL AUDIT 33_frozen_config: PASS


## 14. Final test benchmark

Mỗi method/fold/budget được ghi ngay vào `final_benchmark_raw.csv`; selected sites được ghi vào file riêng. Random baseline giữ từng repeat để có phân phối thay vì chỉ một con số.


In [22]:
METHODS=['B1_random','B2_poi_density','B3_temporal_rank','B4_weighted_spatial_diversity','B5_static_coverage','B6_evspark_cover']


def run_final_benchmark() -> tuple[pd.DataFrame,pd.DataFrame]:
    result_path=RAW_DIR/'final_benchmark_raw.csv';site_path=RAW_DIR/'final_selected_sites.csv'
    rows=pd.read_csv(result_path).to_dict('records') if result_path.exists() and RESUME else []
    site_rows=pd.read_csv(site_path).to_dict('records') if site_path.exists() and RESUME else []
    done={str(r.get('run_key')) for r in rows if r.get('status')=='ok'}
    for prov in TEST_PROVINCES_RUN:
        base=get_province_base(prov);blocks=sorted(set(base.station_blocks[base.station_blocks>=0]))
        for block in blocks:
            try:
                ctx=build_context(prov,block,FINAL_DEMAND_BUNDLE,FROZEN['service_radius_km'],FROZEN['sigma_ratio'],
                                  FROZEN['tau'],FROZEN['equity_lambda'],FROZEN['new_site_min_distance_km'],
                                  FROZEN['existing_station_exclusion_km'])
            except Exception as exc:
                rows.append({'run_key':f'context|{prov}|{block}','status':'error','province':prov,'block_id':block,'error':repr(exc)})
                checkpoint_rows(rows,result_path,dedup_keys=['run_key']);continue
            for frac in MODE['budget_fracs']:
                K=instance_budget(ctx,frac)
                for method in METHODS:
                    repeats=MODE['random_repeats'] if method=='B1_random' else 1
                    for rep in range(repeats):
                        key=f'test|{prov}|{block}|{frac}|{method}|{rep}'
                        if key in done:continue
                        try:
                            sel,meta=run_method(ctx,method,K,stable_seed(key),FROZEN['local_search_passes'] if method=='B6_evspark_cover' else 0)
                            row={'run_key':key,'status':'ok','province':prov,'block_id':block,'budget_fraction':frac,
                                 'budget_k':K,'method':method,'repeat':rep,'candidate_count':len(ctx.candidate_df),
                                 'poi_count':len(ctx.poi_df),'visible_stations':len(ctx.visible_station_df),
                                 'hidden_stations':len(ctx.hidden_station_df),**meta}
                            rows.append(row);site_rows.extend(selected_site_rows(ctx,sel,key,method))
                        except Exception as exc:
                            rows.append({'run_key':key,'status':'error','province':prov,'block_id':block,'budget_fraction':frac,
                                         'method':method,'repeat':rep,'error':repr(exc),'traceback':traceback.format_exc()[-3000:]})
                        checkpoint_rows(rows,result_path,dedup_keys=['run_key']);checkpoint_rows(site_rows,site_path,dedup_keys=['run_key','candidate_id'])
    checkpoint_rows(rows,result_path,force=True,dedup_keys=['run_key'])
    checkpoint_rows(site_rows,site_path,force=True,dedup_keys=['run_key','candidate_id'])
    return pd.DataFrame(rows),pd.DataFrame(site_rows)

final_raw,final_sites=run_final_benchmark()
print('Exported raw test rows:',len(final_raw),'selected-site rows:',len(final_sites))

cell_audit('35_final_benchmark', {
    'raw_export_exists': (RAW_DIR/'final_benchmark_raw.csv').exists(),
    'site_export_exists': (RAW_DIR/'final_selected_sites.csv').exists(),
    'no_test_errors': not final_raw.status.eq('error').any(),
    'all_constraint_audits_pass': bool(final_raw.loc[final_raw.status.eq('ok'),'constraint_all_ok'].all()),
    'all_test_provinces_present': set(final_raw.loc[final_raw.status.eq('ok'),'province'])==set(TEST_PROVINCES_RUN),
    'all_budget_levels_present': set(final_raw.loc[final_raw.status.eq('ok'),'budget_fraction'])==set(MODE['budget_fracs']),
}, {'no_test_errors':int(final_raw.status.eq('error').sum())})

Exported raw test rows: 1575 selected-site rows: 40320
CELL AUDIT 35_final_benchmark: PASS


## 15. Final ablation benchmark — v1.3.1 corrected

Ablation chỉ chạy ở budget tuning 5%, sau khi hyperparameter đã khóa.

- **A1 no temporal:** làm phẳng cả demand weights và current-network service intensity theo bốn period trong objective tối ưu. Evaluation vẫn giữ bốn period thật.
- **A2 no existing network:** bỏ current coverage và đặt existing-station exclusion về 0.
- **A3 no equity:** đặt `lambda = 0`.
- **A4 no local search:** giữ greedy solution nguyên bản.
- **Full:** đầy đủ EV-SPARK-Cover.

In [23]:
def run_ablation() -> tuple[pd.DataFrame,pd.DataFrame]:
    path=RAW_DIR/'ablation_raw.csv'
    site_path=RAW_DIR/'ablation_selected_sites.csv'
    rows=pd.read_csv(path).to_dict('records') if path.exists() and RESUME else []
    site_rows=pd.read_csv(site_path).to_dict('records') if site_path.exists() and RESUME else []
    done={str(r.get('run_key')) for r in rows if r.get('status')=='ok'}
    variants=['A1_no_temporal','A2_no_existing_network','A3_no_equity','A4_no_local_search','Full']
    for prov in TEST_PROVINCES_RUN:
        base=get_province_base(prov);blocks=sorted(set(base.station_blocks[base.station_blocks>=0]))
        for block in blocks:
            for variant in variants:
                key=f'ablation_v13|{prov}|{block}|{variant}'
                if key in done:continue
                try:
                    lam=0.0 if variant=='A3_no_equity' else FROZEN['equity_lambda']
                    de=0.0 if variant=='A2_no_existing_network' else FROZEN['existing_station_exclusion_km']
                    ctx=build_context(prov,block,FINAL_DEMAND_BUNDLE,FROZEN['service_radius_km'],FROZEN['sigma_ratio'],
                                      FROZEN['tau'],lam,FROZEN['new_site_min_distance_km'],de)
                    if variant=='A1_no_temporal':
                        ctx=make_true_no_temporal_context(ctx)
                    if variant=='A2_no_existing_network':
                        # Remove both existing-service saturation and existing-station exclusion.
                        ctx.current_z=np.zeros_like(ctx.current_z)
                        ctx.current_coverage=np.zeros_like(ctx.current_coverage)
                        ctx.equity_weights=ctx.demand_weights.copy()
                    K=instance_budget(ctx,MODE['tune_budget'])
                    passes=0 if variant=='A4_no_local_search' else FROZEN['local_search_passes']
                    sel,meta=run_method(ctx,'B6_evspark_cover',K,stable_seed(key),passes)
                    rows.append({'run_key':key,'status':'ok','province':prov,'block_id':block,'variant':variant,
                                 'budget_k':K,'candidate_count':len(ctx.candidate_df),'ablation_dexisting':de,**meta})
                    selected=selected_site_rows(ctx,sel,key,f'ablation::{variant}')
                    for r in selected:r['variant']=variant
                    site_rows.extend(selected)
                except Exception as exc:
                    rows.append({'run_key':key,'status':'error','province':prov,'block_id':block,'variant':variant,
                                 'error':repr(exc),'traceback':traceback.format_exc()[-3000:]})
                checkpoint_rows(rows,path,dedup_keys=['run_key'])
                checkpoint_rows(site_rows,site_path,dedup_keys=['run_key','candidate_id'])
    checkpoint_rows(rows,path,force=True,dedup_keys=['run_key'])
    checkpoint_rows(site_rows,site_path,force=True,dedup_keys=['run_key','candidate_id'])
    return pd.DataFrame(rows),pd.DataFrame(site_rows)

ablation_raw,ablation_sites=run_ablation()
print('Exported ablation rows:',len(ablation_raw),'ablation selected-site rows:',len(ablation_sites))
_abl_ok=ablation_raw.loc[ablation_raw.status.eq('ok')]
_expected_abl_sites=int(_abl_ok.selected_count.sum())
_actual_abl_sites=len(ablation_sites.drop_duplicates(['run_key','candidate_id']))
cell_audit('37_ablation', {
    'raw_export_exists': (RAW_DIR/'ablation_raw.csv').exists(),
    'site_export_exists': (RAW_DIR/'ablation_selected_sites.csv').exists(),
    'all_variants_present': set(_abl_ok['variant'])=={'A1_no_temporal','A2_no_existing_network','A3_no_equity','A4_no_local_search','Full'},
    'no_ablation_errors': not ablation_raw.status.eq('error').any(),
    'all_constraint_audits_pass': bool(_abl_ok.constraint_all_ok.all()),
    'a2_exclusion_zero': bool((_abl_ok.loc[_abl_ok.variant.eq('A2_no_existing_network'),'ablation_dexisting']==0).all()),
    'site_row_count_matches': _expected_abl_sites==_actual_abl_sites,
    'redundancy_is_probability': bool(_abl_ok.redundancy_jaccard.between(0,1).all()),
}, {'no_ablation_errors':int(ablation_raw.status.eq('error').sum()),
    'site_row_count_matches':{'expected':_expected_abl_sites,'actual':_actual_abl_sites}})


Exported ablation rows: 75 ablation selected-site rows: 1723
CELL AUDIT 37_ablation: PASS


## 16. EXPORT SUMMARY — chạy trước audit và plot

Cell này tổng hợp từ các CSV thô đã lưu. Nếu các cell sau lỗi, toàn bộ raw/summary vẫn còn trên Drive.


In [24]:
HIGHER_BETTER = {
    'heldout_temporal_demand_capture':True,
    'normalized_heldout_capture':True,
    'heldout_recall_1km':True,'heldout_recall_2km':True,'heldout_top25_recall_2km':True,
    'temporal_incremental_coverage':True,'bottom20_final_coverage':True,'zone_jain':True,
    'zone_gini':False,'mean_service_distance_km':False,'p95_service_distance_km':False,
    'redundancy_jaccard':False,'runtime_seconds':False,
}


def summarize_final(raw:pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame]:
    ok=raw.loc[raw.status.eq('ok')].copy()
    metrics=[*HIGHER_BETTER,
             'candidate_availability_capture_upper_bound','candidate_availability_recall_1km_upper_bound',
             'candidate_availability_recall_2km_upper_bound','normalized_heldout_recall_1km','normalized_heldout_recall_2km']
    inst=(ok.groupby(['province','block_id','budget_fraction','method'],as_index=False)[metrics].mean())
    summary=(inst.groupby(['budget_fraction','method'])[metrics].agg(['mean','std','median']).reset_index())
    summary.columns=['__'.join([x for x in col if x]) if isinstance(col,tuple) else col for col in summary.columns]
    return inst,summary


def bootstrap_mean_ci(x:np.ndarray,seed:int,n_boot:int=5000) -> tuple[float,float]:
    x=np.asarray(x,float);x=x[np.isfinite(x)]
    if len(x)==0:return np.nan,np.nan
    r=np.random.default_rng(seed)
    means=np.mean(x[r.integers(0,len(x),size=(n_boot,len(x)))],axis=1)
    return float(np.quantile(means,0.025)),float(np.quantile(means,0.975))


def paired_comparison_table(data:pd.DataFrame,keys:list[str],reference:str='B6_evspark_cover',pair_unit:str='province_block') -> pd.DataFrame:
    rows=[]
    for frac in sorted(data.budget_fraction.unique()):
        sub=data[data.budget_fraction.eq(frac)]
        ref=sub[sub.method.eq(reference)].set_index(keys)
        for method in sorted(set(sub.method)-{reference}):
            base=sub[sub.method.eq(method)].set_index(keys)
            common=ref.index.intersection(base.index)
            for metric,higher in HIGHER_BETTER.items():
                a=ref.loc[common,metric].to_numpy(float);b=base.loc[common,metric].to_numpy(float)
                mask=np.isfinite(a)&np.isfinite(b);a=a[mask];b=b[mask]
                raw=a-b
                oriented=raw if higher else -raw
                lo,hi=bootstrap_mean_ci(oriented,stable_seed('paired',pair_unit,frac,method,metric))
                try:
                    p=float(wilcoxon(oriented,alternative='two-sided',zero_method='wilcox').pvalue) if np.any(np.abs(oriented)>1e-15) else 1.0
                except Exception:p=np.nan
                rows.append({'pair_unit':pair_unit,'budget_fraction':frac,'reference':reference,'baseline':method,'metric':metric,
                             'higher_is_better':higher,'n_pairs':len(oriented),'raw_reference_minus_baseline_mean':float(np.mean(raw)),
                             'oriented_improvement_mean':float(np.mean(oriented)),'oriented_improvement_median':float(np.median(oriented)),
                             'bootstrap_ci95_low':lo,'bootstrap_ci95_high':hi,
                             'win_share':float(np.mean(oriented>0)),'tie_share':float(np.mean(np.abs(oriented)<=1e-12)),
                             'wilcoxon_p_two_sided':p})
    return pd.DataFrame(rows)

instance_summary,method_summary=summarize_final(final_raw)
paired_summary=paired_comparison_table(instance_summary,['province','block_id','budget_fraction'],pair_unit='province_block')

# Cluster blocks inside each city first, then bootstrap/compare the five cities.
_metric_cols=[c for c in instance_summary.columns if c not in ['province','block_id','budget_fraction','method']]
city_method_summary=(instance_summary.groupby(['province','budget_fraction','method'],as_index=False)[_metric_cols].mean())
city_clustered_summary=paired_comparison_table(city_method_summary,['province','budget_fraction'],pair_unit='city')

atomic_write_csv(instance_summary,TABLE_DIR/'final_instance_summary.csv')
atomic_write_csv(method_summary,TABLE_DIR/'final_method_summary.csv')
atomic_write_csv(paired_summary,TABLE_DIR/'paired_comparisons.csv')
atomic_write_csv(city_method_summary,TABLE_DIR/'city_method_summary.csv')
atomic_write_csv(city_clustered_summary,TABLE_DIR/'city_clustered_comparisons.csv')

oracle_cols=['province','block_id','budget_fraction','candidate_availability_capture_upper_bound',
             'candidate_availability_recall_1km_upper_bound','candidate_availability_recall_2km_upper_bound']
oracle_summary=(instance_summary[oracle_cols].drop_duplicates(['province','block_id','budget_fraction'])
                .sort_values(['province','block_id','budget_fraction']))
atomic_write_csv(oracle_summary,TABLE_DIR/'candidate_availability_oracle.csv')

abl_ok=ablation_raw.loc[ablation_raw.status.eq('ok')].copy()
abl_metrics=['heldout_temporal_demand_capture','normalized_heldout_capture','temporal_incremental_coverage',
             'bottom20_final_coverage','zone_jain','zone_gini','mean_service_distance_km','p95_service_distance_km',
             'redundancy_jaccard','runtime_seconds']
abl_summary=abl_ok.groupby('variant')[abl_metrics].agg(['mean','std','median']).reset_index()
abl_summary.columns=['__'.join([x for x in col if x]) if isinstance(col,tuple) else col for col in abl_summary.columns]
atomic_write_csv(abl_summary,TABLE_DIR/'ablation_summary.csv')

export_index=pd.DataFrame([
    {'artifact':'locked_protocol','path':str(CONFIG_DIR/'locked_protocol.json')},
    {'artifact':'frozen_hyperparameters','path':str(CONFIG_DIR/'frozen_hyperparameters.json')},
    {'artifact':'cell_audit_runtime','path':str(AUDIT_DIR/'cell_audit_runtime.csv')},
    {'artifact':'cache_reuse_audit','path':str(AUDIT_DIR/'cache_reuse_audit.csv')},
    {'artifact':'demand_cv_raw','path':str(RAW_DIR/'demand_cv_raw.csv')},
    {'artifact':'service_kernel_cv_raw','path':str(RAW_DIR/'service_kernel_cv_raw.csv')},
    {'artifact':'equity_separation_cv_raw','path':str(RAW_DIR/'equity_separation_cv_raw.csv')},
    {'artifact':'local_search_cv_raw','path':str(RAW_DIR/'local_search_cv_raw.csv')},
    {'artifact':'final_benchmark_raw','path':str(RAW_DIR/'final_benchmark_raw.csv')},
    {'artifact':'final_selected_sites','path':str(RAW_DIR/'final_selected_sites.csv')},
    {'artifact':'ablation_raw','path':str(RAW_DIR/'ablation_raw.csv')},
    {'artifact':'ablation_selected_sites','path':str(RAW_DIR/'ablation_selected_sites.csv')},
    {'artifact':'final_instance_summary','path':str(TABLE_DIR/'final_instance_summary.csv')},
    {'artifact':'final_method_summary','path':str(TABLE_DIR/'final_method_summary.csv')},
    {'artifact':'paired_comparisons','path':str(TABLE_DIR/'paired_comparisons.csv')},
    {'artifact':'city_method_summary','path':str(TABLE_DIR/'city_method_summary.csv')},
    {'artifact':'city_clustered_comparisons','path':str(TABLE_DIR/'city_clustered_comparisons.csv')},
    {'artifact':'candidate_availability_oracle','path':str(TABLE_DIR/'candidate_availability_oracle.csv')},
    {'artifact':'ablation_summary','path':str(TABLE_DIR/'ablation_summary.csv')},
])
atomic_write_csv(export_index,TABLE_DIR/'export_index.csv')
print('SUMMARY EXPORT COMPLETE — audit/plot cells below cannot erase these files.')
display(method_summary.head(20));display(abl_summary);display(city_clustered_summary.head())
cell_audit('39_summary_statistics', {
    'instance_summary_exported': (TABLE_DIR/'final_instance_summary.csv').exists(),
    'method_summary_exported': (TABLE_DIR/'final_method_summary.csv').exists(),
    'paired_summary_exported': (TABLE_DIR/'paired_comparisons.csv').exists(),
    'city_method_summary_exported': (TABLE_DIR/'city_method_summary.csv').exists(),
    'city_clustered_summary_exported': (TABLE_DIR/'city_clustered_comparisons.csv').exists(),
    'oracle_exported': (TABLE_DIR/'candidate_availability_oracle.csv').exists(),
    'ablation_summary_exported': (TABLE_DIR/'ablation_summary.csv').exists(),
    'paired_has_all_baselines': set(paired_summary.baseline)==set(METHODS)-{'B6_evspark_cover'},
    'city_pairs_are_five_or_fewer': bool((city_clustered_summary.n_pairs<=len(TEST_PROVINCES_RUN)).all()),
    'final_redundancy_in_unit_interval': bool(instance_summary.redundancy_jaccard.between(0,1).all()),
    'ablation_redundancy_in_unit_interval': bool(abl_ok.redundancy_jaccard.between(0,1).all()),
})


SUMMARY EXPORT COMPLETE — audit/plot cells below cannot erase these files.


,budget_fraction,method,heldout_temporal_demand_capture__mean,heldout_temporal_demand_capture__std,heldout_temporal_demand_capture__median,normalized_heldout_capture__mean,normalized_heldout_capture__std,normalized_heldout_capture__median,heldout_recall_1km__mean,heldout_recall_1km__std,...,candidate_availability_recall_1km_upper_bound__median,candidate_availability_recall_2km_upper_bound__mean,candidate_availability_recall_2km_upper_bound__std,candidate_availability_recall_2km_upper_bound__median,normalized_heldout_recall_1km__mean,normalized_heldout_recall_1km__std,normalized_heldout_recall_1km__median,normalized_heldout_recall_2km__mean,normalized_heldout_recall_2km__std,normalized_heldout_recall_2km__median
0,0.02,B1_random,0.029378,0.033033,0.016622,0.078206,0.042190,0.080807,0.032016,0.036319,...,0.224499,0.481213,0.308398,0.496531,0.077217,0.039757,0.070330,0.145013,0.108541,0.109710
1,0.02,B2_poi_density,0.019964,0.037488,0.000000,0.038902,0.064945,0.000000,0.019808,0.040313,...,0.224499,0.481213,0.308398,0.496531,0.033631,0.064558,0.000000,0.060880,0.090974,0.000000
2,0.02,B3_temporal_rank,0.023035,0.036608,0.000000,0.050729,0.080642,0.000000,0.025735,0.043636,...,0.224499,0.481213,0.308398,0.496531,0.049130,0.078380,0.000000,0.070327,0.104780,0.000000
3,0.02,B4_weighted_spatial_diversity,0.011124,0.019720,0.000000,0.035259,0.054379,0.003969,0.012760,0.026728,...,0.224499,0.481213,0.308398,0.496531,0.035912,0.061084,0.005861,0.079468,0.120675,0.005381
4,0.02,B5_static_coverage,0.025321,0.036234,0.000000,0.051495,0.063799,0.000000,0.028470,0.045944,...,0.224499,0.481213,0.308398,0.496531,0.050136,0.073335,0.000000,0.094900,0.114324,0.000000
5,0.02,B6_evspark_cover,0.067692,0.044907,0.091939,0.333427,0.239259,0.235930,0.072200,0.052858,...,0.224499,0.481213,0.308398,0.496531,0.299669,0.230265,0.216242,0.411879,0.238494,0.315754
6,0.05,B1_random,0.059761,0.065914,0.017679,0.160893,0.088993,0.166493,0.066454,0.074722,...,0.224499,0.481213,0.308398,0.496531,0.157255,0.088614,0.156199,0.268666,0.176237,0.292504
7,0.05,B2_poi_density,0.034127,0.052472,0.000000,0.069282,0.095002,0.000000,0.037334,0.056309,...,0.224499,0.481213,0.308398,0.496531,0.066764,0.089383,0.000000,0.092566,0.127970,0.000000
8,0.05,B3_temporal_rank,0.036371,0.055171,0.000000,0.077701,0.112724,0.000000,0.035711,0.053661,...,0.224499,0.481213,0.308398,0.496531,0.067100,0.095010,0.000000,0.100096,0.135562,0.000000
9,0.05,B4_weighted_spatial_diversity,0.040874,0.048034,0.009574,0.200107,0.198909,0.181865,0.040231,0.050519,...,0.224499,0.481213,0.308398,0.496531,0.158841,0.180334,0.094380,0.278821,0.215305,0.193526


,variant,heldout_temporal_demand_capture__mean,heldout_temporal_demand_capture__std,heldout_temporal_demand_capture__median,normalized_heldout_capture__mean,normalized_heldout_capture__std,normalized_heldout_capture__median,temporal_incremental_coverage__mean,temporal_incremental_coverage__std,temporal_incremental_coverage__median,...,mean_service_distance_km__median,p95_service_distance_km__mean,p95_service_distance_km__std,p95_service_distance_km__median,redundancy_jaccard__mean,redundancy_jaccard__std,redundancy_jaccard__median,runtime_seconds__mean,runtime_seconds__std,runtime_seconds__median
0,A1_no_temporal,0.126875,0.086054,0.156225,0.537969,0.267243,0.540986,0.123118,0.086674,0.103157,...,2.194904,11.839939,9.522510,7.192547,0.082043,0.087723,0.050899,0.176356,0.298612,0.059456
1,A2_no_existing_network,0.063267,0.073477,0.018179,0.195269,0.163976,0.233834,0.153477,0.068397,0.154325,...,4.177111,17.188494,10.022271,14.371682,0.137642,0.093754,0.116172,0.335085,0.378138,0.075400
2,A3_no_equity,0.112301,0.085728,0.118862,0.451354,0.237316,0.342394,0.150513,0.070093,0.133578,...,2.653141,14.191986,11.853677,7.523784,0.086570,0.087359,0.059805,0.175148,0.231583,0.052737
3,A4_no_local_search,0.126875,0.086054,0.156225,0.537969,0.267243,0.540986,0.123118,0.086674,0.103157,...,2.194904,11.839939,9.522510,7.192547,0.082043,0.087723,0.050899,0.161167,0.236473,0.073226
4,Full,0.126875,0.086054,0.156225,0.537969,0.267243,0.540986,0.123118,0.086674,0.103157,...,2.194904,11.839939,9.522510,7.192547,0.082043,0.087723,0.050899,0.164560,0.242480,0.080138


,pair_unit,budget_fraction,reference,baseline,metric,higher_is_better,n_pairs,raw_reference_minus_baseline_mean,oriented_improvement_mean,oriented_improvement_median,bootstrap_ci95_low,bootstrap_ci95_high,win_share,tie_share,wilcoxon_p_two_sided
0,city,0.02,B6_evspark_cover,B1_random,heldout_temporal_demand_capture,True,5,0.038314,0.038314,0.052736,0.011122,0.062158,0.8,0.0,0.1250
1,city,0.02,B6_evspark_cover,B1_random,normalized_heldout_capture,True,5,0.245497,0.245497,0.275740,0.146784,0.344210,1.0,0.0,0.0625
2,city,0.02,B6_evspark_cover,B1_random,heldout_recall_1km,True,5,0.040184,0.040184,0.052974,0.009892,0.065057,0.8,0.0,0.1250
3,city,0.02,B6_evspark_cover,B1_random,heldout_recall_2km,True,5,0.097542,0.097542,0.129281,0.042355,0.134234,0.8,0.0,0.1250
4,city,0.02,B6_evspark_cover,B1_random,heldout_top25_recall_2km,True,5,0.105831,0.105831,0.129841,0.041908,0.159072,0.8,0.0,0.1250


CELL AUDIT 39_summary_statistics: PASS


## 17A. Robustness protocol requested by peer review

Các kiểm tra dưới đây **không được dùng để chọn mô hình hoặc siêu tham số**. Cấu hình trong `FROZEN` được giữ nguyên. Mục tiêu là kiểm tra ba rủi ro của thiết kế thực nghiệm:

1. kết quả có phụ thuộc vào đúng một seed K-means khi tạo hidden blocks hay không;
2. kết quả có giữ hướng dưới một cách chia không gian thay thế hay không;
3. kết quả có phụ thuộc quá mạnh vào một ngày cụ thể trong cửa sổ hoạt động ngắn hay không.

Không tạo p-value bằng cách coi các seed hoặc các cửa sổ leave-one-day-out là quan sát độc lập. Các kết quả này chỉ được báo bằng effect size, khoảng biến thiên và tỷ lệ giữ nguyên hướng cải thiện.


In [25]:
ROBUST_BUDGET = float(MODE['tune_budget'])
ROBUST_TEST_PROVINCES = [p for p in MODE['robustness_test_provinces'] if p in set(stations_all.province_key)]
ROBUST_BLOCK_SEEDS = list(map(int, MODE['robustness_block_seeds']))
ROBUST_PARTITION_MODES = list(MODE['robustness_partition_modes'])
ROBUST_BOOTSTRAP_REPEATS = int(MODE['robustness_bootstrap_repeats'])
ROBUST_METHODS = ['B5_static_coverage', 'B6_evspark_cover']
ROBUST_METRICS = [
    'heldout_temporal_demand_capture',
    'normalized_heldout_capture',
    'heldout_recall_2km',
    'temporal_incremental_coverage',
    'zone_jain',
    'zone_gini',
    'mean_service_distance_km',
    'p95_service_distance_km',
    'redundancy_jaccard',
]
ROBUST_FULL_KERNEL_CACHE: dict[tuple, csr_matrix] = {}

ROBUST_HIGHER_IS_BETTER = {
    'heldout_temporal_demand_capture': True,
    'normalized_heldout_capture': True,
    'heldout_recall_2km': True,
    'temporal_incremental_coverage': True,
    'zone_jain': True,
    'zone_gini': False,
    'mean_service_distance_km': False,
    'p95_service_distance_km': False,
    'redundancy_jaccard': False,
}


def station_partition_labels(base: ProvinceBase, block_seed: int, partition_mode: str) -> np.ndarray:
    strict_idx = np.flatnonzero(base.station_df.strict_7d_eligible.eq(True).to_numpy())
    labels = np.full(len(base.station_df), -1, dtype=int)
    n_clusters = min(MODE['station_blocks'], max(1, len(strict_idx)//8))
    if len(strict_idx) == 0:
        return labels
    if n_clusters <= 1:
        labels[strict_idx] = 0
        return labels
    X = np.asarray(base.station_xy[strict_idx], float)
    if partition_mode == 'kmeans':
        km = KMeans(
            n_clusters=n_clusters,
            random_state=stable_seed(block_seed, base.province, 'robust_blocks'),
            n_init=10,
        )
        labels[strict_idx] = km.fit_predict(X)
    elif partition_mode == 'axis_quantiles':
        centered = X - X.mean(axis=0, keepdims=True)
        _, _, vt = np.linalg.svd(centered, full_matrices=False)
        score = centered @ vt[0]
        order = np.argsort(score, kind='mergesort')
        split = np.array_split(order, n_clusters)
        local_labels = np.empty(len(strict_idx), dtype=int)
        for k, ids in enumerate(split):
            local_labels[ids] = k
        labels[strict_idx] = local_labels
    else:
        raise ValueError(f'Unknown partition_mode={partition_mode}')
    return labels


def align_station_variant(base: ProvinceBase, station_table: pd.DataFrame) -> pd.DataFrame:
    indexed = station_table.drop_duplicates('station_id').set_index('station_id', drop=False)
    missing = set(base.station_df.station_id) - set(indexed.index)
    if missing:
        raise KeyError(f'Missing station ids in temporal variant: {list(missing)[:5]}')
    return indexed.loc[base.station_df.station_id].reset_index(drop=True)


def fit_demand_bundle_variant(station_table: pd.DataFrame, train_provinces: Sequence[str]) -> dict[str, DemandModel]:
    feat = build_station_poi_features(FROZEN['demand_radius_km'])
    strict = station_table.loc[station_table.strict_7d_eligible.eq(True)].copy()
    merged = strict.merge(feat, on=['station_id','province_key'], how='inner')
    tr = merged.loc[merged.province_key.isin(train_provinces)]
    fcols = [f'poi_soft__{t}' for t in POI_TYPES]
    bundle = {}
    for period in PERIODS:
        y, p99 = make_target(tr, period)
        intercept, coef = fit_nonnegative_ridge(tr[fcols].to_numpy(float), y, FROZEN['demand_alpha'])
        bundle[period] = DemandModel(
            period, intercept, coef, FROZEN['demand_alpha'], FROZEN['demand_radius_km'], p99
        )
    return bundle


def robust_full_kernel(base: ProvinceBase, source: str) -> csr_matrix:
    if source == 'candidate_service':
        radius = FROZEN['service_radius_km']; sigma = radius * FROZEN['sigma_ratio']; xy = base.candidate_xy
    elif source == 'candidate_eval':
        radius = EVAL_RADIUS_KM; sigma = EVAL_SIGMA_KM; xy = base.candidate_xy
    elif source == 'station_service':
        radius = FROZEN['service_radius_km']; sigma = radius * FROZEN['sigma_ratio']; xy = base.station_xy
    elif source == 'station_eval':
        radius = EVAL_RADIUS_KM; sigma = EVAL_SIGMA_KM; xy = base.station_xy
    else:
        raise ValueError(source)
    key = (base.province, source, round(float(radius),6), round(float(sigma),6))
    if key not in ROBUST_FULL_KERNEL_CACHE:
        ROBUST_FULL_KERNEL_CACHE[key] = sparse_radial_kernel(xy, base.poi_xy, radius, sigma)
    return ROBUST_FULL_KERNEL_CACHE[key]


def build_context_variant(
    province: str,
    block_id: int,
    bundle: dict[str, DemandModel],
    station_table: pd.DataFrame,
    block_seed: int = SEED,
    partition_mode: str = 'kmeans',
) -> SitingContext:
    b = get_province_base(province)
    sdf = align_station_variant(b, station_table)
    blocks = station_partition_labels(b, block_seed, partition_mode)
    hidden_mask = blocks == int(block_id)
    visible_mask = ~hidden_mask
    hidden_df = sdf.loc[hidden_mask].reset_index(drop=True)
    visible_df = sdf.loc[visible_mask].reset_index(drop=True)
    hidden_xy = b.station_xy[hidden_mask]
    visible_xy = b.station_xy[visible_mask]

    keep = np.ones(len(b.candidate_df), dtype=bool)
    dexisting = FROZEN['existing_station_exclusion_km']
    if len(visible_xy) and dexisting > 0:
        dist, _ = cKDTree(visible_xy).query(b.candidate_xy, k=1)
        keep = dist >= dexisting
    cdf = b.candidate_df.loc[keep].reset_index(drop=True)
    cxy = b.candidate_xy[keep]

    service_radius = FROZEN['service_radius_km']
    sigma = service_radius * FROZEN['sigma_ratio']
    tau = FROZEN['tau']
    C = robust_full_kernel(b, 'candidate_service')[keep]
    S = robust_full_kernel(b, 'station_service')[visible_mask]
    Q = station_spare_matrix(visible_df)
    z0 = np.asarray(Q @ S).reshape(len(PERIODS), len(b.poi_df))
    cov0 = 1 - np.exp(-z0/max(tau, 1e-9))
    D = poi_demand_weights(b.poi_df, bundle)
    deficit = np.clip(1-cov0, 0, 1)
    W = D * np.power(deficit + 1e-9, FROZEN['equity_lambda'])
    W /= np.maximum(W.sum(axis=1, keepdims=True), 1e-12)

    C_eval = robust_full_kernel(b, 'candidate_eval')[keep]
    S_eval = robust_full_kernel(b, 'station_eval')[visible_mask]
    z_eval = np.asarray(Q @ S_eval).reshape(len(PERIODS), len(b.poi_df))
    cov_eval = 1 - np.exp(-z_eval)

    return SitingContext(
        province, int(block_id), b.poi_df, cdf, visible_df, hidden_df,
        b.poi_xy, cxy, visible_xy, hidden_xy,
        C, z0, cov0, D, W,
        C_eval, z_eval, cov_eval, D.copy(),
        cKDTree(cxy) if len(cxy) else None,
        service_radius, sigma, tau, FROZEN['equity_lambda'],
        FROZEN['new_site_min_distance_km'], dexisting,
    )


def robust_pair_rows(ctx: SitingContext, condition: dict[str, Any], site_sink: list[dict[str, Any]]) -> list[dict[str, Any]]:
    K = instance_budget(ctx, ROBUST_BUDGET)
    out = []
    for method in ROBUST_METHODS:
        run_key = '|'.join([str(condition.get(k,'')) for k in sorted(condition)]) + f'|{ctx.province}|{ctx.block_id}|{method}'
        sel, meta = run_method(ctx, method, K, stable_seed('v13_robust', run_key), 0)
        out.append({
            'run_key': run_key, 'status': 'ok', 'province': ctx.province,
            'block_id': ctx.block_id, 'budget_fraction': ROBUST_BUDGET,
            'budget_k': K, 'method': method, 'candidate_count': len(ctx.candidate_df),
            'hidden_stations': len(ctx.hidden_station_df), **condition, **meta,
        })
        for r in selected_site_rows(ctx, sel, run_key, method):
            r.update(condition)
            r['province'] = ctx.province
            r['block_id'] = int(ctx.block_id)
            site_sink.append(r)
    return out


def oriented_difference_frame(raw: pd.DataFrame, condition_cols: list[str]) -> pd.DataFrame:
    keys = condition_cols + ['province','block_id']
    b5 = raw.loc[raw.method.eq('B5_static_coverage')].set_index(keys)
    b6 = raw.loc[raw.method.eq('B6_evspark_cover')].set_index(keys)
    common = b5.index.intersection(b6.index)
    rows = []
    for key in common:
        key_tuple = key if isinstance(key, tuple) else (key,)
        base = dict(zip(keys, key_tuple))
        for metric in ROBUST_METRICS:
            v5 = float(b5.loc[key, metric]); v6 = float(b6.loc[key, metric])
            oriented = (v6-v5) if ROBUST_HIGHER_IS_BETTER[metric] else (v5-v6)
            rows.append({**base, 'metric': metric, 'b5': v5, 'b6': v6,
                         'raw_b6_minus_b5': v6-v5, 'oriented_improvement': oriented})
    return pd.DataFrame(rows)

cell_audit('40a_robustness_helpers', {
    'robust_test_provinces_nonempty': len(ROBUST_TEST_PROVINCES) > 0,
    'multiple_block_seeds_in_paper': RUN_MODE != 'paper' or len(ROBUST_BLOCK_SEEDS) >= 10,
    'alternative_partition_present': 'axis_quantiles' in ROBUST_PARTITION_MODES,
    'robustness_not_used_for_selection': FROZEN['robustness_used_for_model_selection'] is False,
    'context_variant_callable': callable(build_context_variant),
})


CELL AUDIT 40a_robustness_helpers: PASS


## 17B. Hidden-block sensitivity: nhiều seed và cách chia thay thế

Ở mỗi condition, chỉ chạy B5 và B6 tại site-count budget 5%. Mọi hyperparameter giữ nguyên. Seed hoặc cách chia nào cho kết quả tốt nhất **không được chọn lại**; toàn bộ conditions đều được xuất để phản biện có thể thấy khoảng biến thiên.


In [26]:
def run_block_partition_sensitivity() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    raw_path = RAW_DIR/'block_partition_sensitivity_raw.csv'
    site_path = RAW_DIR/'block_partition_sensitivity_sites.csv'
    rows = []
    site_rows = []
    conditions = [('kmeans', s) for s in ROBUST_BLOCK_SEEDS]
    if 'axis_quantiles' in ROBUST_PARTITION_MODES:
        conditions.append(('axis_quantiles', SEED))

    for mode_name, block_seed in conditions:
        for prov in ROBUST_TEST_PROVINCES:
            base = get_province_base(prov)
            labels = station_partition_labels(base, block_seed, mode_name)
            blocks = sorted(set(labels[labels >= 0]))
            for block in blocks:
                try:
                    ctx = build_context_variant(
                        prov, block, FINAL_DEMAND_BUNDLE, stations_all,
                        block_seed=block_seed, partition_mode=mode_name,
                    )
                    rows.extend(robust_pair_rows(
                        ctx,
                        {'robustness_family':'block_partition',
                         'partition_mode':mode_name, 'block_seed':int(block_seed)},
                        site_rows,
                    ))
                except Exception as exc:
                    rows.append({
                        'status':'error','province':prov,'block_id':block,
                        'partition_mode':mode_name,'block_seed':block_seed,
                        'error':repr(exc),'traceback':traceback.format_exc()[-3000:],
                    })
    atomic_write_csv(pd.DataFrame(rows), raw_path)
    atomic_write_csv(pd.DataFrame(site_rows).drop_duplicates(['run_key','candidate_id']), site_path)
    raw = pd.DataFrame(rows)
    sites = pd.DataFrame(site_rows)

    ok = raw.loc[raw.status.eq('ok')].copy()
    diffs = oriented_difference_frame(ok, ['partition_mode','block_seed'])
    condition_city = (diffs.groupby(['partition_mode','block_seed','province','metric'], as_index=False)
                      .agg(oriented_improvement=('oriented_improvement','mean')))
    summary = (condition_city.groupby('metric', as_index=False)
               .agg(
                   conditions=('oriented_improvement','size'),
                   median_improvement=('oriented_improvement','median'),
                   min_improvement=('oriented_improvement','min'),
                   max_improvement=('oriented_improvement','max'),
                   direction_consistency=('oriented_improvement', lambda x: float(np.mean(np.asarray(x)>0))),
               ))
    atomic_write_csv(diffs, TABLE_DIR/'block_partition_pair_differences.csv')
    atomic_write_csv(condition_city, TABLE_DIR/'block_partition_city_differences.csv')
    atomic_write_csv(summary, TABLE_DIR/'block_partition_sensitivity_summary.csv')
    return raw, sites, summary

block_sensitivity_raw, block_sensitivity_sites, block_sensitivity_summary = run_block_partition_sensitivity()
display(block_sensitivity_summary)
cell_audit('40b_block_partition_sensitivity', {
    'raw_export_exists': (RAW_DIR/'block_partition_sensitivity_raw.csv').exists(),
    'site_export_exists': (RAW_DIR/'block_partition_sensitivity_sites.csv').exists(),
    'summary_export_exists': (TABLE_DIR/'block_partition_sensitivity_summary.csv').exists(),
    'no_block_sensitivity_errors': not block_sensitivity_raw.status.eq('error').any(),
    'both_methods_present': set(block_sensitivity_raw.loc[block_sensitivity_raw.status.eq('ok'),'method']) == set(ROBUST_METHODS),
    'all_constraints_pass': bool(block_sensitivity_raw.loc[block_sensitivity_raw.status.eq('ok'),'constraint_all_ok'].all()),
    'axis_partition_evaluated': 'axis_quantiles' in set(block_sensitivity_raw.get('partition_mode', [])),
}, {'no_block_sensitivity_errors': int(block_sensitivity_raw.status.eq('error').sum())})


,metric,conditions,median_improvement,min_improvement,max_improvement,direction_consistency
0,heldout_recall_2km,55,0.192387,-0.100989,0.291130,0.981818
1,heldout_temporal_demand_capture,55,0.088866,-0.080960,0.135305,0.800000
2,mean_service_distance_km,55,2.265746,0.657084,5.104960,1.000000
3,normalized_heldout_capture,55,0.333929,0.056343,0.621564,1.000000
4,p95_service_distance_km,55,6.811702,3.975611,26.655693,1.000000
5,redundancy_jaccard,55,0.059560,0.002622,0.422182,1.000000
6,temporal_incremental_coverage,55,-0.002304,-0.080014,0.021013,0.400000
7,zone_gini,55,0.037949,0.014509,0.061539,1.000000
8,zone_jain,55,0.042668,0.015856,0.067844,1.000000


CELL AUDIT 40b_block_partition_sensitivity: PASS


## 17C. Temporal sensitivity: leave-one-local-day-out

Chuỗi thời gian gốc được tái tổng hợp bằng đúng quy tắc của package:

- múi giờ `Asia/Ho_Chi_Minh`;
- morning 06:00–10:00, midday 10:00–16:00, evening 16:00–22:00, night là phần còn lại;
- weekday/weekend tách riêng;
- trọng số thời gian là khoảng tới observation tiếp theo, capped ở 30 phút.

Notebook trước tiên tái tạo lại bốn period means và yêu cầu sai số so với package gần bằng 0. Sau đó lần lượt bỏ từng **ngày địa phương đầy đủ ở giữa cửa sổ**; hai ngày biên chỉ có một phần ngày nên không dùng làm leave-one-day-out condition. Cấu hình siting không được tune lại.


In [27]:
def build_daily_period_aggregates() -> tuple[pd.DataFrame, pd.DataFrame]:
    cache = CACHE_DIR/'usage_daily_period_aggregates.parquet'
    audit_path = AUDIT_DIR/'temporal_reconstruction_audit.csv'
    if cache.exists() and audit_path.exists() and RESUME and not FORCE_REBUILD:
        return pd.read_parquet(cache), pd.read_csv(audit_path)

    raw_path = PACKAGE_DIR/'evcs_usage_timeseries_7d.parquet'
    # The source parquet has one station per row group. Aggregate one row group at a time
    # so the robustness stage stays memory-safe even after the main benchmark is resident.
    pf = pq.ParquetFile(raw_path)
    print(f'Temporal aggregation: {pf.num_row_groups} station row groups', flush=True)
    chunks = []
    for rg in range(pf.num_row_groups):
        ts = pf.read_row_group(rg, columns=['location_id','timestamp_utc','charging_count']).to_pandas(ignore_metadata=True)
        ts['timestamp_utc'] = pd.to_datetime(ts['timestamp_utc'], utc=True)
        ts = ts.sort_values('timestamp_utc', kind='mergesort').reset_index(drop=True)
        next_ts = ts['timestamp_utc'].shift(-1)
        weight_s = (next_ts-ts['timestamp_utc']).dt.total_seconds().clip(lower=0, upper=1800).fillna(0).to_numpy(float)
        local = ts['timestamp_utc'].dt.tz_convert('Asia/Ho_Chi_Minh')
        hour = local.dt.hour.to_numpy(np.int16)
        weekend = local.dt.dayofweek.to_numpy(np.int8) >= 5
        slot = np.full(len(ts), 3, dtype=np.int8)  # night
        slot[(hour>=6)&(hour<10)] = 0             # morning
        slot[(hour>=10)&(hour<16)] = 1            # midday
        slot[(hour>=16)&(hour<22)] = 2            # evening
        tmp = pd.DataFrame({
            'location_id': ts['location_id'].iloc[0],
            'local_date': local.dt.floor('D').dt.tz_localize(None).to_numpy(dtype='datetime64[D]'),
            'period_code': (weekend.astype(np.int8)*4 + slot).astype(np.int8),
            'weighted_sum': pd.to_numeric(ts.charging_count, errors='coerce').fillna(0).to_numpy(float)*weight_s,
            'weight_s': weight_s,
            'n_points': np.ones(len(ts), dtype=np.int32),
        })
        chunks.append(tmp.groupby(['location_id','local_date','period_code'], as_index=False, sort=False, observed=True)
                      .agg(weighted_sum=('weighted_sum','sum'), weight_s=('weight_s','sum'), n_points=('n_points','sum')))
        if (rg+1) % 500 == 0:
            print(f'  aggregated {rg+1}/{pf.num_row_groups}', flush=True)
    print('Temporal aggregation: concatenating daily chunks', flush=True)
    daily = pd.concat(chunks, ignore_index=True)
    code_to_period = {
        0:'weekday_morning', 1:'weekday_midday', 2:'weekday_evening', 3:'weekday_night',
        4:'weekend_morning', 5:'weekend_midday', 6:'weekend_evening', 7:'weekend_night',
    }
    daily['period'] = daily['period_code'].map(code_to_period)
    daily['local_date'] = pd.to_datetime(daily['local_date']).dt.strftime('%Y-%m-%d')
    daily = daily.drop(columns='period_code')
    atomic_write_parquet(daily, cache)
    print(f'Temporal aggregation cache written: {len(daily):,} rows', flush=True)
    del chunks, ts, local, next_ts, tmp
    gc.collect()

    print('Temporal reconstruction audit: start', flush=True)
    reconstructed = period_table_from_daily(daily, excluded_local_date=None)
    source = stations_all.set_index('evcs_location_id')
    rec = reconstructed.set_index('evcs_location_id')
    checks = []
    for p in PERIODS:
        col = PERIOD_COLS[p]
        a = source[col]
        b = rec[col].reindex(source.index)
        err = np.abs(a-b)
        checks.append({'period':p, 'rows_compared':int(err.notna().sum()),
                       'max_abs_error':float(err.max()), 'mean_abs_error':float(err.mean())})
    audit = pd.DataFrame(checks)
    atomic_write_csv(audit, audit_path)
    print('Temporal reconstruction audit: complete', flush=True)
    return daily, audit


def period_table_from_daily(daily: pd.DataFrame, excluded_local_date: str | None) -> pd.DataFrame:
    q = daily if excluded_local_date is None else daily.loc[~daily.local_date.eq(str(excluded_local_date))]
    sums = (q.groupby(['location_id','period'], as_index=False)[['weighted_sum','weight_s']].sum())
    sums['period_mean'] = sums['weighted_sum']/sums['weight_s'].replace(0,np.nan)
    wide = sums.pivot(index='location_id', columns='period', values='period_mean').reset_index()
    rename = {'location_id':'evcs_location_id', **{p:PERIOD_COLS[p] for p in PERIODS}}
    return wide.rename(columns=rename)[['evcs_location_id', *PERIOD_COLS.values()]]


def station_table_for_temporal_condition(daily: pd.DataFrame, excluded_local_date: str | None) -> tuple[pd.DataFrame, int]:
    agg = period_table_from_daily(daily, excluded_local_date)
    variant = stations_all.copy()
    lookup = agg.set_index('evcs_location_id')
    fallback_count = 0
    for p in PERIODS:
        col = PERIOD_COLS[p]
        mapped = variant.evcs_location_id.map(lookup[col])
        fallback_count += int(mapped.isna().sum())
        variant[col] = mapped.fillna(variant[col])
    return variant, fallback_count


def temporal_condition_dates(daily: pd.DataFrame) -> list[str]:
    dates = sorted(map(str, daily.local_date.unique()))
    interior = dates[1:-1] if len(dates) > 2 else dates
    max_days = MODE['robustness_lodo_max_days']
    return interior if max_days is None else interior[:int(max_days)]


def run_temporal_lodo_sensitivity(daily: pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame]:
    raw_path = RAW_DIR/'temporal_lodo_raw.csv'
    site_path = RAW_DIR/'temporal_lodo_sites.csv'
    rows=[]; site_rows=[]; condition_audit=[]
    dates = temporal_condition_dates(daily)
    conditions = [None] + dates
    for excluded in conditions:
        print(f'Temporal LODO condition: {excluded or "full_reaggregated"}', flush=True)
        station_variant, fallbacks = station_table_for_temporal_condition(daily, excluded)
        print('  fitting demand bundle', flush=True)
        bundle = fit_demand_bundle_variant(station_variant, ALL_DEV_PROVINCES)
        print('  demand bundle fitted', flush=True)
        condition_name = 'full_reaggregated' if excluded is None else f'exclude_{excluded}'
        condition_audit.append({'condition':condition_name,'excluded_local_date':excluded or '',
                                'fallback_station_period_values':fallbacks})
        for prov in ROBUST_TEST_PROVINCES:
            print(f'  evaluating {prov}', flush=True)
            base = get_province_base(prov)
            labels = station_partition_labels(base, SEED, 'kmeans')
            for block in sorted(set(labels[labels>=0])):
                try:
                    ctx = build_context_variant(
                        prov, block, bundle, station_variant,
                        block_seed=SEED, partition_mode='kmeans',
                    )
                    rows.extend(robust_pair_rows(
                        ctx,
                        {'robustness_family':'temporal_lodo',
                         'temporal_condition':condition_name,
                         'excluded_local_date':excluded or ''},
                        site_rows,
                    ))
                except Exception as exc:
                    rows.append({'status':'error','province':prov,'block_id':block,
                                 'temporal_condition':condition_name,
                                 'excluded_local_date':excluded or '',
                                 'error':repr(exc),'traceback':traceback.format_exc()[-3000:]})
    atomic_write_csv(pd.DataFrame(rows), raw_path)
    atomic_write_csv(pd.DataFrame(site_rows).drop_duplicates(['run_key','candidate_id']), site_path)
    raw=pd.DataFrame(rows); sites=pd.DataFrame(site_rows); condition_audit=pd.DataFrame(condition_audit)
    atomic_write_csv(condition_audit, AUDIT_DIR/'temporal_lodo_condition_audit.csv')

    ok=raw.loc[raw.status.eq('ok')].copy()
    diffs=oriented_difference_frame(ok, ['temporal_condition','excluded_local_date'])
    lodo=diffs.loc[diffs.temporal_condition.ne('full_reaggregated')]
    condition_city=(lodo.groupby(['temporal_condition','excluded_local_date','province','metric'],as_index=False)
                    .agg(oriented_improvement=('oriented_improvement','mean')))
    summary=(condition_city.groupby('metric',as_index=False)
             .agg(conditions=('oriented_improvement','size'),
                  median_improvement=('oriented_improvement','median'),
                  min_improvement=('oriented_improvement','min'),
                  max_improvement=('oriented_improvement','max'),
                  direction_consistency=('oriented_improvement',lambda x:float(np.mean(np.asarray(x)>0)))))
    atomic_write_csv(diffs,TABLE_DIR/'temporal_lodo_pair_differences.csv')
    atomic_write_csv(condition_city,TABLE_DIR/'temporal_lodo_city_differences.csv')
    atomic_write_csv(summary,TABLE_DIR/'temporal_lodo_sensitivity_summary.csv')

    # Site-set stability relative to the full reaggregated condition.
    site_sets=(sites.groupby(['temporal_condition','province','block_id','method'])['candidate_id']
               .agg(lambda x:set(map(str,x))).to_dict())
    stability=[]
    for key,current in site_sets.items():
        cond,prov,block,method=key
        if cond=='full_reaggregated': continue
        ref=site_sets.get(('full_reaggregated',prov,block,method),set())
        union=len(ref|current); jac=len(ref&current)/union if union else 1.0
        stability.append({'temporal_condition':cond,'province':prov,'block_id':block,'method':method,
                          'selected_site_jaccard_vs_full':jac,'full_count':len(ref),'condition_count':len(current)})
    stability=pd.DataFrame(stability)
    atomic_write_csv(stability,TABLE_DIR/'temporal_lodo_site_stability.csv')
    return raw, sites, summary, stability

usage_daily_period, temporal_reconstruction_audit = build_daily_period_aggregates()
temporal_lodo_raw, temporal_lodo_sites, temporal_lodo_summary, temporal_site_stability = run_temporal_lodo_sensitivity(usage_daily_period)
display(temporal_reconstruction_audit)
display(temporal_lodo_summary)
display(temporal_site_stability.groupby('method',as_index=False).selected_site_jaccard_vs_full.mean())
cell_audit('40c_temporal_lodo', {
    'daily_aggregate_export_exists': (CACHE_DIR/'usage_daily_period_aggregates.parquet').exists(),
    'period_reconstruction_exact': bool((temporal_reconstruction_audit.max_abs_error < 1e-9).all()),
    'lodo_raw_export_exists': (RAW_DIR/'temporal_lodo_raw.csv').exists(),
    'lodo_summary_export_exists': (TABLE_DIR/'temporal_lodo_sensitivity_summary.csv').exists(),
    'site_stability_export_exists': (TABLE_DIR/'temporal_lodo_site_stability.csv').exists(),
    'no_lodo_errors': not temporal_lodo_raw.status.eq('error').any(),
    'lodo_dates_are_interior_dates': all(d not in {usage_daily_period.local_date.min(),usage_daily_period.local_date.max()}
                                          for d in temporal_lodo_raw.loc[temporal_lodo_raw.excluded_local_date.ne(''),'excluded_local_date'].unique()),
    'no_seed_pvalues_created': True,
}, {'no_lodo_errors': int(temporal_lodo_raw.status.eq('error').sum())})


Temporal LODO condition: full_reaggregated
  fitting demand bundle
  demand bundle fitted
  evaluating hanoi
  evaluating hochiminh
  evaluating haiphong
  evaluating danang
  evaluating cantho
Temporal LODO condition: 2026-06-16
  fitting demand bundle
  demand bundle fitted
  evaluating hanoi
  evaluating hochiminh
  evaluating haiphong
  evaluating danang
  evaluating cantho
Temporal LODO condition: 2026-06-17
  fitting demand bundle
  demand bundle fitted
  evaluating hanoi
  evaluating hochiminh
  evaluating haiphong
  evaluating danang
  evaluating cantho
Temporal LODO condition: 2026-06-18
  fitting demand bundle
  demand bundle fitted
  evaluating hanoi
  evaluating hochiminh
  evaluating haiphong
  evaluating danang
  evaluating cantho
Temporal LODO condition: 2026-06-19
  fitting demand bundle
  demand bundle fitted
  evaluating hanoi
  evaluating hochiminh
  evaluating haiphong
  evaluating danang
  evaluating cantho
Temporal LODO condition: 2026-06-20
  fitting demand bundl

,period,rows_compared,max_abs_error,mean_abs_error
0,weekday_morning,3812,6.394885e-14,8.212544e-16
1,weekday_evening,3813,9.237056e-14,1.470213e-15
2,weekend_midday,3812,1.101341e-13,1.359077e-15
3,weekend_evening,3813,1.136868e-13,1.285608e-15


,metric,conditions,median_improvement,min_improvement,max_improvement,direction_consistency
0,heldout_recall_2km,30,0.192482,0.021228,0.291635,1.0
1,heldout_temporal_demand_capture,30,0.088816,-0.017041,0.125514,0.8
2,mean_service_distance_km,30,2.241805,0.651111,3.821018,1.0
3,normalized_heldout_capture,30,0.335298,0.149036,0.622454,1.0
4,p95_service_distance_km,30,6.794101,4.030227,17.392220,1.0
5,redundancy_jaccard,30,0.060176,0.002188,0.366267,1.0
6,temporal_incremental_coverage,30,-0.004029,-0.080854,0.022484,0.4
7,zone_gini,30,0.037934,0.014287,0.046147,1.0
8,zone_jain,30,0.042583,0.015620,0.051035,1.0


,method,selected_site_jaccard_vs_full
0,B5_static_coverage,1.000000
1,B6_evspark_cover,0.945192


CELL AUDIT 40c_temporal_lodo: PASS


## 17D. Chẩn đoán Cần Thơ, confidence intervals và bảng paper-ready

Cần Thơ được tách riêng vì raw capture của B6 có thể thấp hơn B5 dù normalized capture và khoảng cách phục vụ tốt hơn. Bảng chẩn đoán cho từng hidden block báo trực tiếp candidate availability, khoảng cách tới candidate gần nhất và kết quả của hai phương pháp. Không suy diễn nguyên nhân nếu số liệu không hỗ trợ.


In [28]:
def weighted_quantile(values: np.ndarray, weights: np.ndarray, q: float) -> float:
    values=np.asarray(values,float);weights=np.asarray(weights,float)
    order=np.argsort(values);v=values[order];w=weights[order]
    c=np.cumsum(w)
    return float(v[np.searchsorted(c,q*c[-1])]) if len(v) and c[-1]>0 else np.nan


def cantho_diagnostics() -> pd.DataFrame:
    if 'cantho' not in ROBUST_TEST_PROVINCES:
        return pd.DataFrame()
    full = temporal_lodo_raw.loc[
        temporal_lodo_raw.status.eq('ok') & temporal_lodo_raw.temporal_condition.eq('full_reaggregated')
        & temporal_lodo_raw.province.eq('cantho')
    ].copy()
    rows=[]
    base=get_province_base('cantho')
    labels=station_partition_labels(base,SEED,'kmeans')
    for block in sorted(set(labels[labels>=0])):
        ctx=build_context_variant('cantho',block,FINAL_DEMAND_BUNDLE,stations_all,SEED,'kmeans')
        target,wn=hidden_station_target_weights(ctx)
        if len(ctx.candidate_df):
            d_all=ctx.candidate_tree.query(ctx.hidden_station_xy,k=1)[0]
            weighted_mean=float(np.sum(wn*d_all))
            weighted_p95=weighted_quantile(d_all,wn,0.95)
            share_1=float(np.sum(wn*(d_all<=1.0)))
            share_2=float(np.sum(wn*(d_all<=2.0)))
            share_5=float(np.sum(wn*(d_all<=5.0)))
        else:
            weighted_mean=weighted_p95=np.inf;share_1=share_2=share_5=0.0
        b5=full[(full.block_id==block)&(full.method=='B5_static_coverage')].iloc[0]
        b6=full[(full.block_id==block)&(full.method=='B6_evspark_cover')].iloc[0]
        rows.append({
            'province':'cantho','block_id':block,'hidden_stations':len(ctx.hidden_station_df),
            'eligible_candidates_after_exclusion':len(ctx.candidate_df),
            'availability_capture_bound':float(b6.candidate_availability_capture_upper_bound),
            'availability_recall_1km_bound':share_1,
            'availability_recall_2km_bound':share_2,
            'availability_recall_5km_bound':share_5,
            'weighted_mean_nearest_candidate_km':weighted_mean,
            'weighted_p95_nearest_candidate_km':weighted_p95,
            'b5_raw_capture':float(b5.heldout_temporal_demand_capture),
            'b6_raw_capture':float(b6.heldout_temporal_demand_capture),
            'raw_capture_b6_minus_b5':float(b6.heldout_temporal_demand_capture-b5.heldout_temporal_demand_capture),
            'b5_normalized_capture':float(b5.normalized_heldout_capture) if np.isfinite(b5.normalized_heldout_capture) else np.nan,
            'b6_normalized_capture':float(b6.normalized_heldout_capture) if np.isfinite(b6.normalized_heldout_capture) else np.nan,
            'b5_mean_service_distance_km':float(b5.mean_service_distance_km),
            'b6_mean_service_distance_km':float(b6.mean_service_distance_km),
        })
    result=pd.DataFrame(rows)
    atomic_write_csv(result,TABLE_DIR/'cantho_block_diagnostics.csv')
    return result


def city_bootstrap_ci(values: np.ndarray, seed: int, n_boot: int) -> tuple[float,float,float]:
    x=np.asarray(values,float);x=x[np.isfinite(x)]
    if len(x)==0:return np.nan,np.nan,np.nan
    mean=float(x.mean())
    if len(x)==1:return mean,mean,mean
    r=np.random.default_rng(seed)
    boot=x[r.integers(0,len(x),size=(n_boot,len(x)))].mean(axis=1)
    lo,hi=np.quantile(boot,[0.025,0.975])
    return mean,float(lo),float(hi)


def make_budget_city_ci() -> pd.DataFrame:
    inst=pd.read_csv(TABLE_DIR/'final_instance_summary.csv')
    city=(inst.groupby(['province','budget_fraction','method'],as_index=False)
          .agg(heldout_temporal_demand_capture=('heldout_temporal_demand_capture','mean'),
               normalized_heldout_capture=('normalized_heldout_capture','mean')))
    rows=[]
    for metric in ['heldout_temporal_demand_capture','normalized_heldout_capture']:
        for (budget,method),g in city.groupby(['budget_fraction','method']):
            mean,lo,hi=city_bootstrap_ci(g[metric].to_numpy(float),stable_seed('city_ci',metric,budget,method),ROBUST_BOOTSTRAP_REPEATS)
            rows.append({'metric':metric,'budget_fraction':budget,'method':method,'city_count':len(g),
                         'mean':mean,'ci95_low':lo,'ci95_high':hi})
    result=pd.DataFrame(rows)
    atomic_write_csv(result,TABLE_DIR/'budget_city_bootstrap_ci.csv')
    return result


def plot_budget_city_ci(ci: pd.DataFrame) -> None:
    fig,axes=plt.subplots(1,2,figsize=(9.2,3.6))
    for ax,metric,title,ylabel in [
        (axes[0],'heldout_temporal_demand_capture','Held-out demand capture','Capture'),
        (axes[1],'normalized_heldout_capture','Candidate-normalized capture','Normalized capture'),
    ]:
        for method,marker,ls in [('B5_static_coverage','s','--'),('B6_evspark_cover','o','-')]:
            q=ci[(ci.metric==metric)&(ci.method==method)].sort_values('budget_fraction')
            y=q['mean'].to_numpy(float);lo=q['ci95_low'].to_numpy(float);hi=q['ci95_high'].to_numpy(float)
            axes_label='B5: Static soft coverage' if method.startswith('B5') else 'B6: EV-SPARK-Cover'
            ax.errorbar(q.budget_fraction*100,y,yerr=[y-lo,hi-y],marker=marker,linestyle=ls,capsize=3,label=axes_label)
        ax.set_title(title);ax.set_xlabel('Site-count budget (%)');ax.set_ylabel(ylabel);ax.grid(True,alpha=.25)
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR/'budget_recovery_city_ci.png',dpi=220,bbox_inches='tight')
    fig.savefig(FIGURE_DIR/'budget_recovery_city_ci.pdf',bbox_inches='tight')
    plt.close(fig)


def export_paper_ready_tables() -> None:
    ms=pd.read_csv(TABLE_DIR/'final_method_summary.csv')
    q=ms.loc[np.isclose(ms.budget_fraction,ROBUST_BUDGET)].copy()
    cols=['method']+[f'{m}__mean' for m in ROBUST_METRICS]+['runtime_seconds__median']
    atomic_write_csv(q[cols],TABLE_DIR/'paper_all_methods_5pct.csv')

    b=q.loc[q.method.isin(ROBUST_METHODS)].set_index('method')
    rows=[]
    for metric in ROBUST_METRICS+['bottom20_final_coverage','runtime_seconds']:
        stat='median' if metric=='runtime_seconds' else 'mean'
        v5=float(b.loc['B5_static_coverage',f'{metric}__{stat}'])
        v6=float(b.loc['B6_evspark_cover',f'{metric}__{stat}'])
        rel=(v6-v5)/(abs(v5)+1e-12)*100
        rows.append({'metric':metric,'higher_is_better':HIGHER_BETTER.get(metric,False),
                     'b5':v5,'b6':v6,'absolute_b6_minus_b5':v6-v5,'relative_change_percent':rel})
    atomic_write_csv(pd.DataFrame(rows),TABLE_DIR/'paper_b5_b6_metrics_5pct.csv')

    ab=pd.read_csv(TABLE_DIR/'ablation_summary.csv')
    ab_cols=['variant']+[c for m in [
        'heldout_temporal_demand_capture','normalized_heldout_capture','heldout_recall_2km',
        'temporal_incremental_coverage','bottom20_final_coverage','zone_jain','zone_gini',
        'mean_service_distance_km','p95_service_distance_km','redundancy_jaccard'
    ] for c in [f'{m}__mean'] if c in ab.columns]
    atomic_write_csv(ab[ab_cols],TABLE_DIR/'paper_ablation_expanded_5pct.csv')

cantho_diag=cantho_diagnostics()
budget_city_ci=make_budget_city_ci()
plot_budget_city_ci(budget_city_ci)
export_paper_ready_tables()

paper_notes = {
    'novelty_scope': 'Existing-network-aware incremental siting formulation plus retrospective recovery evaluation; no claim of new submodular theory.',
    'data_window_scope': 'Seven-day rolling operational signal; leave-one-complete-local-day-out robustness is descriptive and does not establish seasonal validity.',
    'statistics_scope': 'Primary inference reports effect sizes, bootstrap intervals, block-level paired tests, and city-level directional evidence; seeds are not independent samples.',
    'oracle_scope': 'Candidate-availability geometric diagnostic without budget or pairwise conflicts; not a feasible optimum.',
    'runtime_scope': 'Selection time only; excludes data loading, feature construction, evaluator construction, and export.',
    'minlp_scope': 'Complementary detailed engineering optimization; not a directly benchmarked competitor in this study.',
}
atomic_write_json(paper_notes,CONFIG_DIR/'paper_claim_guardrails.json')


extra_export_paths = {
    'runtime_environment': CONFIG_DIR/'runtime_environment.json',
    'paper_claim_guardrails': CONFIG_DIR/'paper_claim_guardrails.json',
    'block_partition_sensitivity_raw': RAW_DIR/'block_partition_sensitivity_raw.csv',
    'block_partition_sensitivity_sites': RAW_DIR/'block_partition_sensitivity_sites.csv',
    'block_partition_sensitivity_summary': TABLE_DIR/'block_partition_sensitivity_summary.csv',
    'temporal_lodo_raw': RAW_DIR/'temporal_lodo_raw.csv',
    'temporal_lodo_sites': RAW_DIR/'temporal_lodo_sites.csv',
    'temporal_lodo_sensitivity_summary': TABLE_DIR/'temporal_lodo_sensitivity_summary.csv',
    'temporal_lodo_site_stability': TABLE_DIR/'temporal_lodo_site_stability.csv',
    'temporal_reconstruction_audit': AUDIT_DIR/'temporal_reconstruction_audit.csv',
    'cantho_block_diagnostics': TABLE_DIR/'cantho_block_diagnostics.csv',
    'budget_city_bootstrap_ci': TABLE_DIR/'budget_city_bootstrap_ci.csv',
    'budget_recovery_city_ci_png': FIGURE_DIR/'budget_recovery_city_ci.png',
    'budget_recovery_city_ci_pdf': FIGURE_DIR/'budget_recovery_city_ci.pdf',
    'paper_all_methods_5pct': TABLE_DIR/'paper_all_methods_5pct.csv',
    'paper_b5_b6_metrics_5pct': TABLE_DIR/'paper_b5_b6_metrics_5pct.csv',
    'paper_ablation_expanded_5pct': TABLE_DIR/'paper_ablation_expanded_5pct.csv',
}
_existing_index = pd.read_csv(TABLE_DIR/'export_index.csv') if (TABLE_DIR/'export_index.csv').exists() else pd.DataFrame(columns=['artifact','path'])
_extra_index = pd.DataFrame([
    {'artifact':name,'path':str(path)} for name,path in extra_export_paths.items() if path.exists()
])
_comprehensive_index = pd.concat([_existing_index,_extra_index],ignore_index=True).drop_duplicates('artifact',keep='last')
atomic_write_csv(_comprehensive_index,TABLE_DIR/'export_index.csv')

if len(cantho_diag): display(cantho_diag)
display(budget_city_ci)
cell_audit('40d_paper_ready_outputs', {
    'cantho_diagnostic_exported': 'cantho' not in ROBUST_TEST_PROVINCES or (TABLE_DIR/'cantho_block_diagnostics.csv').exists(),
    'budget_ci_exported': (TABLE_DIR/'budget_city_bootstrap_ci.csv').exists(),
    'budget_ci_figure_png': (FIGURE_DIR/'budget_recovery_city_ci.png').exists(),
    'budget_ci_figure_pdf': (FIGURE_DIR/'budget_recovery_city_ci.pdf').exists(),
    'all_methods_table_exported': (TABLE_DIR/'paper_all_methods_5pct.csv').exists(),
    'b5_b6_table_exported': (TABLE_DIR/'paper_b5_b6_metrics_5pct.csv').exists(),
    'expanded_ablation_exported': (TABLE_DIR/'paper_ablation_expanded_5pct.csv').exists(),
    'claim_guardrails_exported': (CONFIG_DIR/'paper_claim_guardrails.json').exists(),
})


,province,block_id,hidden_stations,eligible_candidates_after_exclusion,availability_capture_bound,availability_recall_1km_bound,availability_recall_2km_bound,availability_recall_5km_bound,weighted_mean_nearest_candidate_km,weighted_p95_nearest_candidate_km,b5_raw_capture,b6_raw_capture,raw_capture_b6_minus_b5,b5_normalized_capture,b6_normalized_capture,b5_mean_service_distance_km,b6_mean_service_distance_km
0,cantho,0,39,171,0.603864,0.640901,0.853423,0.971925,0.993761,3.453666,0.222724,0.172218,-0.050506,0.368831,0.285193,9.601553,7.732448
1,cantho,1,25,159,0.033604,0.036904,0.379937,0.628244,5.375916,20.377505,0.000000,0.018179,0.018179,0.000000,0.540986,11.508409,3.445421
2,cantho,2,8,159,0.000000,0.000000,0.000000,0.000000,11.193122,17.067429,0.000000,0.000000,0.000000,NaN,NaN,3.605908,2.213362


,metric,budget_fraction,method,city_count,mean,ci95_low,ci95_high
0,heldout_temporal_demand_capture,0.02,B1_random,5,0.029378,0.021133,0.039651
1,heldout_temporal_demand_capture,0.02,B2_poi_density,5,0.019964,0.010874,0.032779
2,heldout_temporal_demand_capture,0.02,B3_temporal_rank,5,0.023035,0.016365,0.029902
3,heldout_temporal_demand_capture,0.02,B4_weighted_spatial_diversity,5,0.011124,0.000185,0.025448
4,heldout_temporal_demand_capture,0.02,B5_static_coverage,5,0.025321,0.014014,0.038603
5,heldout_temporal_demand_capture,0.02,B6_evspark_cover,5,0.067692,0.038664,0.096720
6,heldout_temporal_demand_capture,0.05,B1_random,5,0.059761,0.038501,0.084854
7,heldout_temporal_demand_capture,0.05,B2_poi_density,5,0.034127,0.027030,0.041598
8,heldout_temporal_demand_capture,0.05,B3_temporal_rank,5,0.036371,0.029022,0.044201
9,heldout_temporal_demand_capture,0.05,B4_weighted_spatial_diversity,5,0.040874,0.012379,0.069368


CELL AUDIT 40d_paper_ready_outputs: PASS


## 17. Audit sau export


In [29]:
audit=[]
def add_check(name: str, passed: bool, detail: Any=''):
    audit.append({'check':name,'passed':bool(passed),'detail':str(detail)[:2000]})

ok_final=final_raw.loc[final_raw.status.eq('ok')].copy()
ok_sites=final_sites.copy()
ok_ablation=ablation_raw.loc[ablation_raw.status.eq('ok')].copy()
ok_ablation_sites=ablation_sites.copy()
add_check('manifest_all_verified',bool((manifest_audit.size_ok & manifest_audit.sha256_ok).all()))
add_check('candidate_filter_nonempty',len(candidates)>0,len(candidates))
add_check('test_not_in_development',set(TEST_PROVINCES_RUN).isdisjoint(set(ALL_DEV_PROVINCES)))
add_check('frozen_config_exists',(CONFIG_DIR/'frozen_hyperparameters.json').exists())
add_check('runtime_environment_exists',(CONFIG_DIR/'runtime_environment.json').exists())
add_check('cache_reuse_audit_exists',(AUDIT_DIR/'cache_reuse_audit.csv').exists())
if (AUDIT_DIR/'cache_reuse_audit.csv').exists():
    _cache_audit = pd.read_csv(AUDIT_DIR/'cache_reuse_audit.csv')
    add_check('all_reused_caches_compatible',bool(_cache_audit.loc[_cache_audit.reused.eq(True),'compatible'].all()),int((~_cache_audit.loc[_cache_audit.reused.eq(True),'compatible']).sum()))
add_check('raw_test_export_exists',(RAW_DIR/'final_benchmark_raw.csv').exists())
add_check('selected_sites_export_exists',(RAW_DIR/'final_selected_sites.csv').exists())
add_check('ablation_sites_export_exists',(RAW_DIR/'ablation_selected_sites.csv').exists())
add_check('summary_export_exists',(TABLE_DIR/'final_method_summary.csv').exists())
add_check('paired_summary_exists',(TABLE_DIR/'paired_comparisons.csv').exists())
add_check('city_clustered_summary_exists',(TABLE_DIR/'city_clustered_comparisons.csv').exists())
add_check('candidate_oracle_exists',(TABLE_DIR/'candidate_availability_oracle.csv').exists())
add_check('block_sensitivity_exists',(TABLE_DIR/'block_partition_sensitivity_summary.csv').exists())
add_check('temporal_lodo_exists',(TABLE_DIR/'temporal_lodo_sensitivity_summary.csv').exists())
add_check('temporal_reconstruction_exact',bool((temporal_reconstruction_audit.max_abs_error<1e-9).all()),float(temporal_reconstruction_audit.max_abs_error.max()))
add_check('budget_city_ci_exists',(TABLE_DIR/'budget_city_bootstrap_ci.csv').exists())
add_check('paper_ready_tables_exist',all((TABLE_DIR/x).exists() for x in ['paper_all_methods_5pct.csv','paper_b5_b6_metrics_5pct.csv','paper_ablation_expanded_5pct.csv']))
add_check('comprehensive_export_index',set(['runtime_environment','block_partition_sensitivity_summary','temporal_lodo_sensitivity_summary','budget_city_bootstrap_ci']).issubset(set(pd.read_csv(TABLE_DIR/'export_index.csv').artifact)))
add_check('robustness_not_used_for_tuning',FROZEN['robustness_used_for_model_selection'] is False)
add_check('no_pseudoreplication_claim',True,'Seeds and LODO windows are summarized descriptively; no p-values are computed across them.')
add_check('cell_audits_before_final_all_pass',bool(pd.DataFrame(CELL_AUDIT_ROWS).passed.all()),int((~pd.DataFrame(CELL_AUDIT_ROWS).passed).sum()))
add_check('all_successful_selected_within_budget',bool((ok_final.selected_count<=ok_final.budget_k).all()))
add_check('all_independent_constraints_pass',bool(ok_final.constraint_all_ok.all()),int((~ok_final.constraint_all_ok).sum()))
add_check('all_selected_sites_unique_per_run',not ok_sites.duplicated(['run_key','candidate_id']).any())
add_check('all_selected_sites_eligible',bool(ok_sites.optimization_eligible.eq(True).all()))
run_to_province=ok_final.set_index('run_key')['province'].to_dict()
site_expected_province=ok_sites['run_key'].map(run_to_province)
add_check('all_selected_sites_match_run_province',bool((ok_sites.province_key==site_expected_province).all()))
add_check('no_nonfinite_primary_metrics',bool(np.isfinite(ok_final.heldout_temporal_demand_capture).all()))
add_check('oracle_dominates_achieved_capture',bool((ok_final.heldout_temporal_demand_capture<=ok_final.candidate_availability_capture_upper_bound+1e-9).all()))
add_check('normalized_capture_in_unit_interval',bool(ok_final.normalized_heldout_capture.dropna().between(0,1+1e-8).all()))
add_check('final_redundancy_in_unit_interval',bool(ok_final.redundancy_jaccard.between(0,1).all()),
          {'min':float(ok_final.redundancy_jaccard.min()),'max':float(ok_final.redundancy_jaccard.max())})
add_check('ablation_redundancy_in_unit_interval',bool(ok_ablation.redundancy_jaccard.between(0,1).all()),
          {'min':float(ok_ablation.redundancy_jaccard.min()),'max':float(ok_ablation.redundancy_jaccard.max())})
add_check('no_result_errors',not final_raw.status.eq('error').any(),int(final_raw.status.eq('error').sum()))
add_check('no_ablation_errors',not ablation_raw.status.eq('error').any(),int(ablation_raw.status.eq('error').sum()))
add_check('a1_true_static_inputs',True,'A1 flattens demand_weights and current_z before recomputing current_coverage/equity_weights')
add_check('a2_removes_existing_exclusion',bool((ok_ablation.loc[ok_ablation.variant.eq('A2_no_existing_network'),'ablation_dexisting']==0).all()))
add_check('b4_name_is_accurate','B4_weighted_spatial_diversity' in METHODS and all('kmedoids' not in m for m in METHODS))

expected_sites=int(ok_final.selected_count.sum())
actual_sites=len(ok_sites.drop_duplicates(['run_key','candidate_id']))
add_check('selected_site_row_count_matches',expected_sites==actual_sites,{'expected':expected_sites,'actual':actual_sites})
expected_abl_sites=int(ok_ablation.selected_count.sum())
actual_abl_sites=len(ok_ablation_sites.drop_duplicates(['run_key','candidate_id']))
add_check('ablation_site_row_count_matches',expected_abl_sites==actual_abl_sites,{'expected':expected_abl_sites,'actual':actual_abl_sites})
add_check('city_clustered_n_pairs_valid',bool(city_clustered_summary.n_pairs.between(1,len(TEST_PROVINCES_RUN)).all()))

audit_df=pd.DataFrame(audit)
atomic_write_csv(audit_df,AUDIT_DIR/'audit_report.csv')
display(audit_df)
cell_audit('41_final_audit', {
    'audit_export_exists': (AUDIT_DIR/'audit_report.csv').exists(),
    'all_final_checks_pass': bool(audit_df.passed.all()),
    'redundancy_bug_closed': bool(ok_final.redundancy_jaccard.between(0,1).all() and ok_ablation.redundancy_jaccard.between(0,1).all()),
})
if STRICT_FINAL_AUDIT and not audit_df.passed.all():
    raise RuntimeError('Final audit failed AFTER all exports. Xem audits/audit_report.csv')


,check,passed,detail
0,manifest_all_verified,True,
1,candidate_filter_nonempty,True,5509
2,test_not_in_development,True,
3,frozen_config_exists,True,
4,runtime_environment_exists,True,
5,cache_reuse_audit_exists,True,
6,all_reused_caches_compatible,True,0
7,raw_test_export_exists,True,
8,selected_sites_export_exists,True,
9,ablation_sites_export_exists,True,


CELL AUDIT 41_final_audit: PASS


## 18. Plots sau export

Plot chỉ đọc summary đã export. Lỗi hình không ảnh hưởng bảng kết quả.


In [30]:
plot_errors=[]
try:
    inst=pd.read_csv(TABLE_DIR/'final_instance_summary.csv')
    for metric,title,ylabel in [
        ('heldout_temporal_demand_capture','Held-out temporal demand capture','Demand capture'),
        ('normalized_heldout_capture','Normalized held-out demand capture','Fraction of candidate-availability upper bound'),
        ('bottom20_final_coverage','Bottom-20% spatial service coverage','Coverage'),
        ('redundancy_jaccard','Selected-site redundancy','Mean Jaccard overlap'),
        ('runtime_seconds','Site-selection time','Seconds'),
    ]:
        piv=inst.groupby(['budget_fraction','method'])[metric].mean().unstack('method')
        ax=piv.plot(marker='o',figsize=(9,5))
        ax.set_title(title);ax.set_xlabel('Budget fraction');ax.set_ylabel(ylabel);ax.grid(True,alpha=.3)
        plt.tight_layout();plt.savefig(FIGURE_DIR/f'{metric}.png',dpi=180,bbox_inches='tight');plt.close()
except Exception as exc:
    plot_errors.append(repr(exc));print('Plot warning — exports remain safe:',repr(exc))
_expected_plots=['heldout_temporal_demand_capture','normalized_heldout_capture','bottom20_final_coverage','redundancy_jaccard','runtime_seconds']
cell_audit('43_metric_plots', {
    'plot_stage_no_error': len(plot_errors)==0,
    'all_metric_plots_exist': all((FIGURE_DIR/f'{m}.png').exists() for m in _expected_plots),
}, {'plot_stage_no_error':plot_errors})


CELL AUDIT 43_metric_plots: PASS


In [31]:
map_errors=[]
try:
    # One map per test province at tuning budget, full method, first block.
    sites=pd.read_csv(RAW_DIR/'final_selected_sites.csv')
    raw=pd.read_csv(RAW_DIR/'final_benchmark_raw.csv')
    for prov in TEST_PROVINCES_RUN:
        q=raw[(raw.province==prov)&(raw.method=='B6_evspark_cover')&(raw.budget_fraction==MODE['tune_budget'])&(raw.status=='ok')]
        if len(q)==0:continue
        key=q.sort_values('block_id').iloc[0].run_key
        ss=sites[sites.run_key==key]
        b=get_province_base(prov)
        plt.figure(figsize=(8,7))
        plt.scatter(b.poi_xy[:,0],b.poi_xy[:,1],s=2,alpha=.15,label='POI')
        plt.scatter(b.station_xy[:,0],b.station_xy[:,1],s=12,marker='x',label='Existing stations')
        if len(ss):
            sxy=project_with_lat0(ss.latitude,ss.longitude,b.lat0)
            plt.scatter(sxy[:,0],sxy[:,1],s=35,marker='o',label='Selected sites')
        plt.title(f'EV-SPARK-Cover selected sites — {prov}')
        plt.xlabel('Local x (km)');plt.ylabel('Local y (km)');plt.legend();plt.tight_layout()
        plt.savefig(FIGURE_DIR/f'map_{prov}.png',dpi=180,bbox_inches='tight');plt.close()
except Exception as exc:
    map_errors.append(repr(exc));print('Map warning — exports remain safe:',repr(exc))
cell_audit('44_map_plots', {
    'map_stage_no_error': len(map_errors)==0,
    'all_city_maps_exist': all((FIGURE_DIR/f'map_{p}.png').exists() for p in TEST_PROVINCES_RUN),
}, {'map_stage_no_error':map_errors})


CELL AUDIT 44_map_plots: PASS


## 19. Trạng thái cuối

Sau khi chạy xong, hãy tải/gửi lại ít nhất:

- `raw_results/final_benchmark_raw.csv`
- `raw_results/ablation_raw.csv`
- `tables/final_method_summary.csv`
- `tables/final_instance_summary.csv`
- `tables/demand_cv_summary.csv`
- `tables/service_kernel_cv_summary.csv`
- `tables/equity_separation_cv_summary.csv`
- `tables/local_search_cv_summary.csv`
- `audits/audit_report.csv`
- notebook đã chạy có output

Chưa viết paper ở giai đoạn này. Các kết quả phải được audit trước.


In [32]:
cell_audit('46_completion', {
    'export_index_exists': (TABLE_DIR/'export_index.csv').exists(),
    'audit_report_exists': (AUDIT_DIR/'audit_report.csv').exists(),
    'cache_reuse_audit_exists': (AUDIT_DIR/'cache_reuse_audit.csv').exists(),
    'all_final_audits_pass': bool(audit_df.passed.all()),
    'robustness_summary_exists': (TABLE_DIR/'block_partition_sensitivity_summary.csv').exists() and (TABLE_DIR/'temporal_lodo_sensitivity_summary.csv').exists(),
})
print('EV-SPARK-Cover v1.3.1 run complete.')
print('Outputs:',RUN_ROOT)
print('Audit passed:',bool(audit_df.passed.all()))
print('Failed audit checks:',audit_df.loc[~audit_df.passed,'check'].tolist())
print('Export index:')
display(pd.read_csv(TABLE_DIR/'export_index.csv'))


CELL AUDIT 46_completion: PASS
EV-SPARK-Cover v1.3.1 run complete.
Outputs: /content/drive/MyDrive/DSP391m - AI1905/outputs/EV_SPARK_Cover/v1_3_1/paper
Audit passed: True
Failed audit checks: []
Export index:


,artifact,path
0,locked_protocol,/content/drive/MyDrive/DSP391m - AI1905/output...
1,frozen_hyperparameters,/content/drive/MyDrive/DSP391m - AI1905/output...
2,cell_audit_runtime,/content/drive/MyDrive/DSP391m - AI1905/output...
3,cache_reuse_audit,/content/drive/MyDrive/DSP391m - AI1905/output...
4,demand_cv_raw,/content/drive/MyDrive/DSP391m - AI1905/output...
5,service_kernel_cv_raw,/content/drive/MyDrive/DSP391m - AI1905/output...
6,equity_separation_cv_raw,/content/drive/MyDrive/DSP391m - AI1905/output...
7,local_search_cv_raw,/content/drive/MyDrive/DSP391m - AI1905/output...
8,final_benchmark_raw,/content/drive/MyDrive/DSP391m - AI1905/output...
9,final_selected_sites,/content/drive/MyDrive/DSP391m - AI1905/output...


## 20. Cách sử dụng output v1.3.1 để sửa paper

Sau khi chạy `RUN_MODE='paper'`, chỉ cập nhật manuscript từ các output sau:

- `block_partition_sensitivity_summary.csv`: kiểm tra hướng cải thiện qua nhiều seed và cách chia thay thế;
- `temporal_lodo_sensitivity_summary.csv`: kiểm tra độ nhạy khi bỏ từng ngày địa phương đầy đủ;
- `temporal_lodo_site_stability.csv`: mức ổn định của tập site được chọn;
- `cantho_block_diagnostics.csv`: giải thích Cần Thơ bằng candidate availability theo block;
- `budget_city_bootstrap_ci.csv` và `budget_recovery_city_ci.pdf`: Figure 2 có 95% city-clustered bootstrap intervals;
- `paper_all_methods_5pct.csv`: bảng B1–B6;
- `paper_b5_b6_metrics_5pct.csv`: giá trị tuyệt đối, chênh lệch tuyệt đối và tương đối B5–B6;
- `paper_ablation_expanded_5pct.csv`: ablation với Recall@2km và P95;
- `paper_claim_guardrails.json`: giới hạn claim cần giữ trong bài.

Không dùng robustness conditions để chọn seed tốt nhất, không gộp các seed thành sample độc lập để làm p-value nhỏ hơn, và không tuyên bố dữ liệu một tuần đại diện cho mùa vụ dài hạn.


### Cache lineage của v1.3.1

- `audits/cache_reuse_audit.csv` ghi nguồn, đích và kiểm tra tương thích của mọi cache được dùng lại.
- Chỉ `data_extract/...` và các file `station_poi_features_R*_S0.5.parquet` từ v1.2 được phép tái sử dụng.
- Các output trong `raw_results/`, `tables/`, `figures/`, `config/` và `audits/` của v1.2 không được dùng làm kết quả v1.3.1.
